# Kalshi BTC Arb Strategy — Live + Paper Trading Bot

Backtested **+$43.76 over 7 days** vs the existing bot's **-$8.04** (using the same 7-day capture).

| Tier | Strategy | Edge | Risk |
|------|----------|------|------|
| 1 | Monotonicity arb — `yes_bid(K_hi) > yes_ask(K_lo)` | avg 14.9¢/pair | **Risk-free** |
| 2 | Deep-ITM convergence — `price ≥ 0.88, fair ≥ 0.94, ttc ∈ [15,60]min` | avg 2¢/contract | 98% win rate |

| § | Section | Purpose |
|---|---------|--------|
| 1 | Config & Auth | Kalshi REST client, credentials, trading params |
| 2 | Live Data | Coinbase spot poller, Kalshi websocket, event tracker, tick logger |
| 3 | Signal Detection | Tier 1 monotonicity scan + Tier 2 deep-ITM scan |
| 4 | Execution | Single + paired order placement, position tracking |
| 5 | Main Loop | Start/stop orchestrator |
| 6 | Controls | Status, diagnostics, kill switch, enable_live |


In [ ]:
%pip install -q 'websockets>=13' duckdb


In [8]:
# § 1 — Config, imports, Kalshi API client
from __future__ import annotations
import asyncio, base64, hashlib, inspect, json, math, os, sqlite3, sys, threading, time, uuid, warnings
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Optional, List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import requests
import websockets

# version-safe websockets header param detection
_ws_sig = inspect.signature(websockets.connect)
_WS_HEADER_PARAM = 'additional_headers' if 'additional_headers' in _ws_sig.parameters else 'extra_headers'

warnings.filterwarnings('ignore')

# ── Trading config ────────────────────────────────────────────────
CFG = {
    'mode': 'paper',                   # 'paper' or 'live'
    'live_enabled': False,

    # ─ Tier 1 (monotonicity arb — RISK-FREE) ─
    # V3 backtest: median book depth on T1 candidates is 17 (p95 52). Raising
    # cap from 10 to 20 captures more of the available depth on each arb.
    # V5 backtest: T1 in CALM spot regimes (|spot_move_30s| ≤ $30) prints
    # 100% win at avg qty 13.2 over 7d (t=3.28). Calm regimes likely persist
    # longer in live execution (less HFT competition).
    't1_enabled': True,
    't1_min_net_edge_cents': 1.5,
    't1_max_qty_per_leg': 20,         # was 10 (V3 depth-headroom finding)
    't1_max_dollars_per_pair': 40.0,  # was 20 (scaled with qty)
    't1_calm_filter_enabled': True,   # V5 — only trade T1 in calm spot regimes
    't1_calm_max_spot_move_30s': 30.0,  # dollars; |spot(t) - spot(t-30s)|

    # ─ Tier 2 (deep-ITM convergence — STATISTICAL) ─
    # Backtest H22: price≥0.88, fair≥0.95, edge≥1.5c, ttc 30-60m, spread≤2c
    #   → 71 trades, 98.6% win, +$8.99 unlimited capital
    #   → +$8.87 on $100 with 2× caps
    't2_enabled': True,
    't2_min_price': 0.88,
    't2_min_fair': 0.95,
    't2_min_edge_cents': 1.5,
    't2_min_dollar_distance_from_strike': 300.0,
    't2_min_pct_distance_from_strike': 0.004,
    't2_max_spread_at_entry': 0.02,
    't2_min_secs_to_close': 1800,
    't2_max_secs_to_close': 3600,
    't2_max_qty_per_strike': 5,       # was 2
    't2_max_dollars_per_trade': 15.0, # was 6
    't2_sigma_floor_annual': 0.35,

    # ─ Tier 3 (OTM persistence NO — NEW, validated out-of-sample) ─
    # Spot has been < strike for ≥5 min, buy NO at $0.60-$0.72 (yes_bid 0.28-0.40)
    # Backtest: 53 trades, 79.2% win, +$8.83. Asymmetric (only NO works on this sample).
    # IMPORTANT: edge is sample-specific to May 2026 BTC trend. Cap small.
    't3_enabled': True,
    't3_min_yes_bid': 0.28,
    't3_max_yes_bid': 0.40,
    't3_min_strike_distance': 100.0,
    't3_persistence_seconds': 300,    # spot must have been below K for ≥5 min
    't3_min_secs_to_close': 600,      # 10 min
    't3_max_secs_to_close': 1800,     # 30 min
    't3_max_qty_per_strike': 5,
    't3_max_dollars_per_trade': 12.0,

    # ─ Portfolio risk ─
    'max_concurrent_positions': 4,
    'max_total_exposure': 200.0,
    'daily_loss_limit': -30.0,

    # ─ Fees ─
    'kalshi_fee_cap': 0.07,

    # ─ Polling / scheduling ─
    'spot_poll_sec': 2.0,
    'decision_interval_sec': 3.0,
    'sigma_window_min': 60,
    'sigma_min_points': 15,
    'sigma_uncertainty_discount': 0.50,

    # ─ Websocket ─
    'ws_url': 'wss://external-api-ws.kalshi.com/trade-api/ws/v2',
    'ws_reconnect_base_sec': 2.0,
    'ws_reconnect_max_sec': 60.0,

    # ─ Order execution ─
    'order_buffer_cents': 1,
    'order_expiration_sec': 20,
    'order_post_timeout_sec': 3.0,

    # ─ Data persistence ─
    'trades_db_path': 'output/arb_strategy_trades.db',
    'ticks_db_path': 'output/arb_strategy_ticks.db',

    # ─ Universe ─
    # IMPORTANT — only KXBTCD (threshold-style: "BTC closes above $K") works
    # for this strategy. KXBTC (range/bucket: "BTC closes in $K–$K+100") has
    # a bell-curve price structure where monotonicity is not arbitrage:
    # both legs could end up worth $0, and a 90c loss is possible.
    'event_series': ('KXBTCD',),
}

Path('output').mkdir(exist_ok=True)


def kalshi_fee(price: float) -> float:
    p = max(0.0, min(1.0, price))
    return min(CFG['kalshi_fee_cap'], 0.07 * p / 0.50)


def _norm_cdf(x: float) -> float:
    return 0.5 * math.erfc(-x / math.sqrt(2))


# ── Kalshi REST + Auth Client ─────────────────────────────────────
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding


def load_credentials(env_path='~/.kalshi/credentials.env'):
    creds = {}
    path = Path(env_path).expanduser()
    if not path.exists():
        print(f'Warning: no credentials file at {path}')
        return creds
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        creds[k.strip()] = v.strip()
    return creds


class KalshiClient:
    PROD_URL = 'https://api.elections.kalshi.com/trade-api/v2'
    DEMO_URL = 'https://demo-api.kalshi.co/trade-api/v2'

    def __init__(self, env='prod', key_id=None, private_key_path=None):
        assert env in ('prod', 'demo')
        self.env = env
        self.base_url = self.PROD_URL if env == 'prod' else self.DEMO_URL
        self._path_prefix = '/trade-api/v2'

        creds = load_credentials()
        prefix = 'KALSHI_PROD_' if env == 'prod' else 'KALSHI_DEMO_'
        self.key_id = key_id or creds.get(prefix + 'KEY_ID')
        kp = private_key_path or creds.get(prefix + 'PRIVATE_KEY_PATH')
        self.private_key = None
        if kp:
            kp_path = Path(kp).expanduser()
            if kp_path.exists():
                with open(kp_path, 'rb') as f:
                    self.private_key = serialization.load_pem_private_key(f.read(), password=None)
        self.session = requests.Session()

    def _sign(self, method, path):
        if not self.private_key or not self.key_id:
            return {}
        ts = str(int(time.time() * 1000))
        msg = (ts + method + path.split('?')[0]).encode('utf-8')
        sig = self.private_key.sign(
            msg,
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
            hashes.SHA256())
        return {
            'KALSHI-ACCESS-KEY': self.key_id,
            'KALSHI-ACCESS-SIGNATURE': base64.b64encode(sig).decode(),
            'KALSHI-ACCESS-TIMESTAMP': ts,
        }

    def ws_auth_headers(self):
        return self._sign('GET', '/trade-api/ws/v2')

    def _get(self, path, params=None):
        url = self.base_url + path
        headers = self._sign('GET', self._path_prefix + path)
        r = self.session.get(url, headers=headers, params=params, timeout=10)
        r.raise_for_status()
        return r.json()

    def get_events(self, series_ticker=None, status='open', limit=100):
        params = {'status': status, 'limit': limit}
        if series_ticker:
            params['series_ticker'] = series_ticker
        return self._get('/events', params)

    def get_markets(self, event_ticker=None, status='open', limit=200):
        params = {'status': status, 'limit': limit}
        if event_ticker:
            params['event_ticker'] = event_ticker
        return self._get('/markets', params)

    def get_market(self, ticker):
        return self._get(f'/markets/{ticker}')

    def get_orderbook(self, ticker, depth=10):
        return self._get(f'/markets/{ticker}/orderbook', {'depth': depth})

    def get_balance(self):
        return self._get('/portfolio/balance')

    def get_positions(self):
        return self._get('/portfolio/positions')

    def place_order(self, ticker, side, action, count, yes_price_cents, expiration_sec=20):
        # Limit order. side in {'yes','no'}, action in {'buy','sell'}.
        body = {
            'ticker': ticker, 'side': side, 'action': action, 'count': count,
            'type': 'limit',
            'client_order_id': f'arb-{uuid.uuid4().hex[:16]}',
            f'{side}_price': int(round(yes_price_cents)),
            'expiration_ts': int(time.time()) + expiration_sec,
        }
        h = self._sign('POST', self._path_prefix + '/portfolio/orders')
        h['Content-Type'] = 'application/json'
        r = self.session.post(self.base_url + '/portfolio/orders',
                              headers=h, json=body, timeout=10)
        if r.status_code >= 400:
            return {'error': r.text[:200], 'status': r.status_code}
        return r.json()

    def cancel_order(self, order_id):
        h = self._sign('DELETE', self._path_prefix + f'/portfolio/orders/{order_id}')
        r = self.session.delete(self.base_url + f'/portfolio/orders/{order_id}',
                                headers=h, timeout=10)
        return {'status': r.status_code, 'body': r.text[:200]}


# ── Set up clients ────────────────────────────────────────────────
kalshi_prod = KalshiClient(env='prod')
kalshi_prod.private_key = None
kalshi_prod.key_id = None

kalshi_live = None
try:
    _kl = KalshiClient(env='prod')
    if _kl.private_key and _kl.key_id:
        bal = _kl.get_balance()
        balance_cents = bal.get('balance', 0) if isinstance(bal, dict) else 0
        print(f'Live auth OK. Balance: ${balance_cents/100:.2f}')
        kalshi_live = _kl
    else:
        print('No prod credentials — paper mode only.')
except Exception as e:
    print(f'Live auth failed: {e}')

try:
    test = kalshi_prod.get_events(series_ticker='KXBTCD', status='open', limit=3)
    print(f'Prod market data OK — {len(test.get("events", []))} events')
except Exception as e:
    print(f'Prod market data failed: {e}')

print(f'Mode: {CFG["mode"]}  T1: {CFG["t1_enabled"]}  T2: {CFG["t2_enabled"]}')
print(f'WS: {CFG["ws_url"]}  (param: {_WS_HEADER_PARAM}, websockets {websockets.__version__})')


Live auth OK. Balance: $97.76
Prod market data OK — 3 events
Mode: paper  T1: True  T2: True
WS: wss://external-api-ws.kalshi.com/trade-api/ws/v2  (param: extra_headers, websockets 13.1)


In [ ]:
# § 2 — Live Data Pipeline
#
# Workers (background threads):
#   1. Coinbase BTC spot poller (every 2s)
#   2. Kalshi websocket listener (ticker channel) → updates BOOKS in-memory
#   3. Event tracker — finds nearest open hourly KXBTCD event
#   4. Tick flusher — persists every tick to SQLite

_LOCK = threading.Lock()

SPOT = {'price': None, 'ts': None, 'history': []}
BOOKS: Dict[str, Dict[str, Any]] = {}
TRACKED = {'event': None, 'close_time': None, 'refreshed_at': None}
BOT_STATE = {'running': False, 'threads': [], 'log': [],
             'trades_this_session': 0, 'iter': 0}

_WS_STATE = {'connected': False, 'subscribed_event': None,
             'reconnect_count': 0, 'last_msg_ts': None, 'msg_count': 0,
             'needs_resubscribe': False, 'mode': 'websocket'}


def _log(msg):
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S')
    entry = f'[{ts}] {msg}'
    BOT_STATE['log'].append(entry)
    if len(BOT_STATE['log']) > 500:
        BOT_STATE['log'] = BOT_STATE['log'][-200:]


# ── Tick logger ──────────────────────────────────────────────────

TICK_INIT_SQL = (
    "CREATE TABLE IF NOT EXISTS ticks ("
    "id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT NOT NULL,"
    " event_ticker TEXT, market_ticker TEXT NOT NULL,"
    " yes_bid REAL, yes_bid_qty REAL, yes_ask REAL, yes_ask_qty REAL,"
    " no_bid REAL, no_ask REAL, volume REAL, source TEXT DEFAULT 'ws',"
    " btc_spot REAL, secs_to_close REAL);"
    "CREATE TABLE IF NOT EXISTS spot_ticks ("
    "id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT NOT NULL,"
    " price REAL NOT NULL, source TEXT);"
    "CREATE INDEX IF NOT EXISTS idx_ticks_ts ON ticks(ts);"
    "CREATE INDEX IF NOT EXISTS idx_ticks_event ON ticks(event_ticker, ts);"
    "CREATE INDEX IF NOT EXISTS idx_ticks_market ON ticks(market_ticker, ts);"
    "CREATE INDEX IF NOT EXISTS idx_spot_ts ON spot_ticks(ts);"
)


def _init_tick_db():
    conn = sqlite3.connect(CFG['ticks_db_path'])
    # WAL = better concurrent reads + faster writes + crash safety for long runs.
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    conn.executescript(TICK_INIT_SQL)
    # Best-effort migration for older DBs that pre-date btc_spot/secs_to_close.
    for col in ('btc_spot REAL', 'secs_to_close REAL'):
        try:
            conn.execute(f'ALTER TABLE ticks ADD COLUMN {col}')
        except sqlite3.OperationalError:
            pass  # already exists
    conn.commit(); conn.close()

_init_tick_db()

_TICK_QUEUE: list = []
_SPOT_TICK_QUEUE: list = []
_TICK_LOCK = threading.Lock()


def _queue_tick(event_ticker, market_ticker, yb, ya, yb_qty, ya_qty, no_bid, no_ask, volume, source='ws'):
    now = datetime.now(timezone.utc)
    ts = now.isoformat()
    # Snapshot spot + secs_to_close at tick time so future backtests have a
    # self-contained row. Read WITHOUT _LOCK — _queue_tick is called from
    # inside _LOCK-holding contexts (e.g. _seed_books_rest) so acquiring _LOCK
    # here would deadlock. Dict reads are safe under the GIL; slight staleness
    # on btc_spot/secs_to_close is fine for a tick log.
    spot = SPOT.get('price')
    ct = TRACKED.get('close_time')
    secs = (ct - now).total_seconds() if ct else None
    with _TICK_LOCK:
        _TICK_QUEUE.append((ts, event_ticker, market_ticker, yb, yb_qty, ya, ya_qty,
                            no_bid, no_ask, volume, source, spot, secs))


def _queue_spot_tick(price, source='coinbase'):
    ts = datetime.now(timezone.utc).isoformat()
    with _TICK_LOCK:
        _SPOT_TICK_QUEUE.append((ts, price, source))


def _tick_flusher():
    while BOT_STATE['running']:
        _flush_ticks()
        _sleep(5.0)
    _flush_ticks()


def _flush_ticks():
    with _TICK_LOCK:
        ticks = list(_TICK_QUEUE); _TICK_QUEUE.clear()
        spots = list(_SPOT_TICK_QUEUE); _SPOT_TICK_QUEUE.clear()
    if not ticks and not spots:
        return
    try:
        conn = sqlite3.connect(CFG['ticks_db_path'])
        if ticks:
            conn.executemany(
                'INSERT INTO ticks(ts,event_ticker,market_ticker,yes_bid,yes_bid_qty,'
                'yes_ask,yes_ask_qty,no_bid,no_ask,volume,source,btc_spot,secs_to_close) '
                'VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)', ticks)
        if spots:
            conn.executemany(
                'INSERT INTO spot_ticks(ts,price,source) VALUES(?,?,?)', spots)
        conn.commit(); conn.close()
    except Exception as e:
        _log(f'tick flush err: {e}')


def tick_stats():
    conn = sqlite3.connect(CFG['ticks_db_path'])
    n_ticks = conn.execute('SELECT COUNT(*) FROM ticks').fetchone()[0]
    n_spots = conn.execute('SELECT COUNT(*) FROM spot_ticks').fetchone()[0]
    first = conn.execute('SELECT MIN(ts) FROM ticks').fetchone()[0]
    last = conn.execute('SELECT MAX(ts) FROM ticks').fetchone()[0]
    n_events = conn.execute('SELECT COUNT(DISTINCT event_ticker) FROM ticks').fetchone()[0]
    n_markets = conn.execute('SELECT COUNT(DISTINCT market_ticker) FROM ticks').fetchone()[0]
    # Coverage of the new backtest-critical columns (NULL if pre-upgrade rows)
    n_spot_col = conn.execute('SELECT COUNT(*) FROM ticks WHERE btc_spot IS NOT NULL').fetchone()[0]
    n_ttc_col  = conn.execute('SELECT COUNT(*) FROM ticks WHERE secs_to_close IS NOT NULL').fetchone()[0]
    # Last-hour throughput (rough health check)
    last_hour = conn.execute(
        "SELECT COUNT(*) FROM ticks WHERE ts > datetime('now','-1 hour')"
    ).fetchone()[0]
    size_mb = os.path.getsize(CFG['ticks_db_path']) / (1024 ** 2)
    conn.close()
    print(f'Ticks: {n_ticks:,}  spot ticks: {n_spots:,}')
    print(f'Events: {n_events}  Markets: {n_markets}')
    print(f'Range: {first or "—"} → {last or "—"}')
    print(f'DB size: {size_mb:.1f} MB')
    print(f'Last hour throughput: {last_hour:,} ticks')
    if n_ticks:
        pct_spot = 100 * n_spot_col / n_ticks
        pct_ttc  = 100 * n_ttc_col  / n_ticks
        print(f'Backtest-ready rows: btc_spot {pct_spot:.0f}%  secs_to_close {pct_ttc:.0f}%')


def snapshot_ticks_db(label: str = None) -> str:
    """Copy the live ticks DB to a timestamped immutable snapshot.

    Call this at the end of an overnight run to lock in a backtestable file.
    Uses SQLite VACUUM INTO so the snapshot is consistent even while the
    bot keeps writing.
    """
    if label is None:
        label = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    out_dir = Path('output/captures'); out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'ticks_capture_{label}.db'
    src = sqlite3.connect(CFG['ticks_db_path'])
    try:
        src.execute(f"VACUUM INTO '{out_path}'")
    finally:
        src.close()
    size_mb = out_path.stat().st_size / (1024 ** 2)
    print(f'Snapshot → {out_path}  ({size_mb:.1f} MB)')
    return str(out_path)


# ── Spot poller ───────────────────────────────────────────────────

def _coinbase_spot():
    try:
        r = requests.get('https://api.coinbase.com/v2/prices/BTC-USD/spot', timeout=5)
        r.raise_for_status()
        return float(r.json()['data']['amount']), 'coinbase'
    except Exception:
        r = requests.get('https://api.coingecko.com/api/v3/simple/price',
                         params={'ids': 'bitcoin', 'vs_currencies': 'usd'}, timeout=5)
        r.raise_for_status()
        return float(r.json()['bitcoin']['usd']), 'coingecko'


def _spot_poller():
    while BOT_STATE['running']:
        try:
            price, source = _coinbase_spot()
            # Hampel outlier filter (defined in §2b — only invoke if available)
            try:
                if 'spot_is_outlier' in globals() and spot_is_outlier(price):
                    _log(f'spot outlier rejected: ${price:.2f} (Hampel filter)')
                    _sleep(CFG['spot_poll_sec'])
                    continue
            except NameError:
                pass
            now = datetime.now(timezone.utc)
            with _LOCK:
                SPOT['price'] = price; SPOT['ts'] = now
                SPOT['history'].append((now, price))
                cutoff = now - timedelta(minutes=CFG['sigma_window_min'] + 10)
                SPOT['history'] = [(t, p) for t, p in SPOT['history'] if t > cutoff]
            _queue_spot_tick(price, source)
            # Update online math models if loaded
            try:
                ewma_update(price); drift_update(price)
            except NameError:
                pass
        except Exception as e:
            _log(f'spot err: {e}')
        _sleep(CFG['spot_poll_sec'])


# ── Kalshi websocket ──────────────────────────────────────────────

def _parse_ticker_msg(msg_data):
    tk = msg_data.get('market_ticker')
    if not tk:
        return
    def _d(key):
        v = msg_data.get(key)
        if v is None or v == '':
            return None
        try: return float(v)
        except (ValueError, TypeError): return None

    ya = _d('yes_ask_dollars'); yb = _d('yes_bid_dollars')
    yb_qty = _d('yes_bid_qty') or _d('volume_fp')
    ya_qty = _d('yes_ask_qty')
    na = (1.0 - yb) if yb is not None else None
    nb = (1.0 - ya) if ya is not None else None
    now = datetime.now(timezone.utc)

    with _LOCK:
        existing = BOOKS.get(tk, {})
        BOOKS[tk] = {
            'yes_bid': yb if yb is not None else existing.get('yes_bid'),
            'yes_ask': ya if ya is not None else existing.get('yes_ask'),
            'yes_bid_qty': yb_qty if yb_qty is not None else existing.get('yes_bid_qty'),
            'yes_ask_qty': ya_qty if ya_qty is not None else existing.get('yes_ask_qty'),
            'no_bid': nb if nb is not None else existing.get('no_bid'),
            'no_ask': na if na is not None else existing.get('no_ask'),
            'floor': existing.get('floor'),
            'close_time': existing.get('close_time'),
            'status': existing.get('status', 'active'),
            'ts': now,
            'volume': msg_data.get('volume_fp') or existing.get('volume'),
        }
        event = TRACKED.get('event')
        _WS_STATE['last_msg_ts'] = now
        _WS_STATE['msg_count'] += 1

    _queue_tick(event, tk, yb, ya, yb_qty, ya_qty, nb, na,
                msg_data.get('volume_fp'), 'ws')


def _seed_books_rest(event_ticker):
    # REST hydrate BOOKS with floor strikes + initial quotes
    try:
        mkts = kalshi_prod.get_markets(event_ticker=event_ticker, limit=200).get('markets', [])
        now = datetime.now(timezone.utc)
        with _LOCK:
            for m in mkts:
                tk = m.get('ticker')
                if not tk:
                    continue
                yb = m.get('yes_bid'); ya = m.get('yes_ask')
                yb = float(yb) / 100 if yb is not None else None
                ya = float(ya) / 100 if ya is not None else None
                na = (1.0 - yb) if yb is not None else None
                nb = (1.0 - ya) if ya is not None else None
                BOOKS[tk] = {
                    'yes_bid': yb, 'yes_ask': ya,
                    'yes_bid_qty': None, 'yes_ask_qty': None,
                    'no_bid': nb, 'no_ask': na,
                    'floor': m.get('floor_strike'),
                    'close_time': m.get('close_time'),
                    'status': (m.get('status') or '').lower(),
                    'ts': now, 'volume': m.get('volume'),
                }
                _queue_tick(event_ticker, tk, yb, ya, None, None, nb, na,
                            m.get('volume'), 'rest')
        _log(f'REST seed: {len(mkts)} markets for {event_ticker}')
    except Exception as e:
        _log(f'REST seed err: {e}')


def _get_event_tickers(event_ticker):
    try:
        mkts = kalshi_prod.get_markets(event_ticker=event_ticker, limit=200).get('markets', [])
        return [m['ticker'] for m in mkts if m.get('ticker')]
    except Exception:
        return []


async def _try_subscribe(ws, reason='', sub_id=1):
    """Try to send a subscribe message for the currently-tracked event.
    Idempotent: only sends if we have an event AND it's not already our sub.
    Returns True if subscribed."""
    with _LOCK:
        event = TRACKED.get('event')
    if not event:
        return False
    if event == _WS_STATE.get('subscribed_event'):
        return True
    tickers = _get_event_tickers(event)
    if not tickers:
        _log(f'WS subscribe skipped ({reason}): no tickers for {event}')
        return False
    try:
        await ws.send(json.dumps({
            'id': sub_id, 'cmd': 'subscribe',
            'params': {'channels': ['ticker'], 'market_tickers': tickers}}))
    except Exception as e:
        _log(f'WS subscribe send err ({reason}): {e}')
        return False
    _WS_STATE['subscribed_event'] = event
    _log(f'WS subscribed ({reason}): {len(tickers)} tickers on {event}')
    # Hydrate via REST in parallel so we have book data immediately
    _seed_books_rest(event)
    return True


async def _ws_connect():
    while BOT_STATE['running']:
        if kalshi_live is None:
            _log('WS: no auth client — REST fallback')
            _WS_STATE['mode'] = 'rest_fallback'
            return
        auth_headers = kalshi_live.ws_auth_headers()
        if not auth_headers:
            _log('WS: no auth headers — REST fallback')
            _WS_STATE['mode'] = 'rest_fallback'
            return
        try:
            async with websockets.connect(CFG['ws_url'], **{_WS_HEADER_PARAM: auth_headers}) as ws:
                _WS_STATE['connected'] = True
                _WS_STATE['reconnect_count'] = 0
                _WS_STATE['mode'] = 'websocket'
                _log('WS connected')

                # Initial subscription attempt
                await _try_subscribe(ws, reason='initial')

                sub_id_counter = [10]
                last_sub_attempt = time.time()

                while BOT_STATE['running']:
                    # Resubscribe edge — event changed
                    if _WS_STATE['needs_resubscribe']:
                        _WS_STATE['needs_resubscribe'] = False
                        await _try_subscribe(ws, reason='event-changed', sub_id=sub_id_counter[0])
                        sub_id_counter[0] += 1
                    # Retry subscription every 8s if we still have no sub yet
                    elif _WS_STATE.get('subscribed_event') is None and (time.time() - last_sub_attempt) > 8.0:
                        await _try_subscribe(ws, reason='retry', sub_id=sub_id_counter[0])
                        sub_id_counter[0] += 1
                        last_sub_attempt = time.time()
                    try:
                        raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                    except asyncio.TimeoutError:
                        continue
                    if not raw:
                        continue
                    try:
                        data = json.loads(raw)
                    except json.JSONDecodeError:
                        continue
                    mt = data.get('type', '')
                    if mt == 'ticker':
                        _parse_ticker_msg(data.get('msg', {}))
                    elif mt == 'subscribed':
                        sid = data.get('msg', {}).get('sid')
                        _log(f'WS sub confirmed, sid={sid}')
                    elif mt == 'error':
                        _log(f'WS error: {data.get("msg", {})}')

        except (websockets.exceptions.InvalidStatusCode, websockets.exceptions.InvalidStatus) as e:
            code_attr = getattr(e, 'status_code', None) or getattr(getattr(e, 'response', None), 'status_code', None)
            _log(f'WS rejected: HTTP {code_attr}')
            if code_attr in (401, 403):
                _log('WS auth failed — REST fallback')
                _WS_STATE['mode'] = 'rest_fallback'
                _WS_STATE['connected'] = False
                return
        except Exception as e:
            _log(f'WS err: {e}')

        _WS_STATE['connected'] = False
        _WS_STATE['subscribed_event'] = None
        if not BOT_STATE['running']:
            break
        _WS_STATE['reconnect_count'] += 1
        delay = min(CFG['ws_reconnect_base_sec'] * (2 ** _WS_STATE['reconnect_count']),
                    CFG['ws_reconnect_max_sec'])
        _log(f'WS reconnecting in {delay:.0f}s')
        await asyncio.sleep(delay)


def _ws_listener():
    loop = asyncio.new_event_loop()
    try:
        loop.run_until_complete(_ws_connect())
    except Exception as e:
        _log(f'WS loop err: {e}')
    finally:
        loop.close()
    if _WS_STATE['mode'] == 'rest_fallback':
        _rest_poller()


def _rest_poller():
    _log('REST poller active (6s)')
    while BOT_STATE['running']:
        with _LOCK:
            event = TRACKED.get('event')
        if not event:
            _sleep(2); continue
        try:
            mkts = kalshi_prod.get_markets(event_ticker=event, limit=200).get('markets', [])
            now = datetime.now(timezone.utc)
            with _LOCK:
                for m in mkts:
                    tk = m.get('ticker')
                    if not tk: continue
                    yb = m.get('yes_bid'); ya = m.get('yes_ask')
                    yb = float(yb)/100 if yb is not None else None
                    ya = float(ya)/100 if ya is not None else None
                    na = (1.0 - yb) if yb is not None else None
                    nb = (1.0 - ya) if ya is not None else None
                    BOOKS[tk] = {
                        'yes_bid': yb, 'yes_ask': ya,
                        'yes_bid_qty': None, 'yes_ask_qty': None,
                        'no_bid': nb, 'no_ask': na,
                        'floor': m.get('floor_strike'),
                        'close_time': m.get('close_time'),
                        'status': (m.get('status') or '').lower(),
                        'ts': now, 'volume': m.get('volume')}
                    _queue_tick(event, tk, yb, ya, None, None, nb, na, m.get('volume'), 'rest')
                _WS_STATE['last_msg_ts'] = now
                _WS_STATE['msg_count'] += len(mkts)
        except Exception as e:
            _log(f'REST poll err: {e}')
        _sleep(6.0)


# ── Event tracker ─────────────────────────────────────────────────

def _scan_for_event():
    """One pass over all configured series; returns (event_ticker, close_time)
    or (None, None). Logs candidate counts."""
    from dateutil import parser as dtparser
    now = datetime.now(timezone.utc)
    best_event = None; best_close = None; total_candidates = 0
    for series in CFG['event_series']:
        try:
            resp = kalshi_prod.get_events(series_ticker=series, status='open', limit=50)
        except Exception as e:
            _log(f'tracker fetch err ({series}): {e}')
            continue
        events = resp.get('events', [])
        total_candidates += len(events)
        for ev in events:
            et = ev.get('event_ticker', '')
            ct_str = None
            for m in ev.get('markets', []) or []:
                ct_str = m.get('close_time') or m.get('expected_expiration_time')
                if ct_str: break
            if not ct_str:
                try:
                    mr = kalshi_prod.get_markets(event_ticker=et, status='open', limit=5)
                    for m in mr.get('markets', []):
                        ct_str = m.get('close_time') or m.get('expected_expiration_time')
                        if ct_str: break
                except Exception:
                    pass
            if not ct_str: continue
            ct = dtparser.isoparse(ct_str)
            if ct.tzinfo is None: ct = ct.replace(tzinfo=timezone.utc)
            ttl = (ct - now).total_seconds()
            if ttl < 120 or ttl > 7200: continue
            if best_close is None or ct < best_close:
                best_event = et; best_close = ct
    return best_event, best_close, total_candidates


def _apply_tracked_event(best_event, best_close):
    """Update TRACKED + trigger resubscribe if event changed.

    If best_event is None (e.g. transient API failure), keep the current
    tracked event UNLESS its close_time has already passed."""
    with _LOCK:
        old = TRACKED.get('event')
        old_close = TRACKED.get('close_time')
        new_is_better = best_event and best_event != old
        # Keep current event if API gave us nothing but our event is still alive
        if best_event is None and old is not None and old_close is not None:
            if (old_close - datetime.now(timezone.utc)).total_seconds() > 0:
                TRACKED['refreshed_at'] = datetime.now(timezone.utc)
                return  # keep what we have
        if new_is_better:
            _log(f'Tracking: {best_event} closes {best_close}')
            BOOKS.clear()
            _WS_STATE['needs_resubscribe'] = True
        TRACKED['event'] = best_event
        TRACKED['close_time'] = best_close
        TRACKED['refreshed_at'] = datetime.now(timezone.utc)
    if new_is_better:
        _seed_books_rest(best_event)


def _event_tracker():
    while BOT_STATE['running']:
        try:
            best_event, best_close, n_cand = _scan_for_event()
            if best_event is None and n_cand > 0:
                _log(f'tracker: {n_cand} events found but none in 2min–2hr window')
            elif best_event is None:
                _log(f'tracker: no open events for {CFG["event_series"]}')
            _apply_tracked_event(best_event, best_close)
        except Exception as e:
            _log(f'tracker err: {e}')
        _sleep(60)


# ── Helpers ───────────────────────────────────────────────────────

def _sleep(secs):
    end = time.time() + secs
    while time.time() < end and BOT_STATE['running']:
        time.sleep(0.2)


def _causal_sigma():
    # Annualized BTC vol from recent spot history (causal — only past data)
    with _LOCK:
        hist = list(SPOT['history'])
    if len(hist) < CFG['sigma_min_points']:
        return None
    prices = np.array([p for _, p in hist], dtype=float)
    lr = np.diff(np.log(prices))
    if len(lr) < 5: return None
    s = float(np.std(lr) * np.sqrt(525960))
    if not np.isfinite(s) or s <= 0: return None
    return s


print('Data pipeline ready. tick_stats() to inspect collected data.')


In [9]:
# § 2b — Mathematical models (classical quant, no ML)
#
# Stacks on top of §2's data pipeline. Provides:
#   - EWMA volatility estimator (RiskMetrics 1996) — adapts faster than rolling
#   - Welford running stats (Welford 1962) — numerically stable mean/variance
#   - Hampel filter (median + MAD) for spot outlier rejection
#   - Black-Scholes binary with drift correction
#   - Merton jump-diffusion overlay (handles fat tails)
#   - Kelly fraction position sizing
#   - SPRT (Wald 1947) for per-tier auto-disable on declining win rate
#
# All deterministic, no hidden state across runs (except SPRT which persists
# its log-likelihood ratio to the trades DB).


# ─────────────────────────────────────────────────────────────────
# 1. EWMA volatility estimator (RiskMetrics)
#    σ²_t = λ × σ²_{t-1} + (1−λ) × r²_t
#    λ = 0.94 is the standard RiskMetrics value (~ 75-period half-life)
# ─────────────────────────────────────────────────────────────────

EWMA_STATE = {'sigma_sq': None, 'last_price': None, 'lambda': 0.94, 'n_updates': 0}


def ewma_update(new_price: float):
    """Update EWMA σ estimate with a new spot price."""
    if EWMA_STATE['last_price'] is None:
        EWMA_STATE['last_price'] = new_price
        return
    r = math.log(new_price / EWMA_STATE['last_price'])
    EWMA_STATE['last_price'] = new_price
    EWMA_STATE['n_updates'] += 1
    if EWMA_STATE['sigma_sq'] is None:
        EWMA_STATE['sigma_sq'] = r * r
    else:
        l = EWMA_STATE['lambda']
        EWMA_STATE['sigma_sq'] = l * EWMA_STATE['sigma_sq'] + (1 - l) * r * r


def ewma_sigma_annual() -> Optional[float]:
    """Return annualized σ from EWMA estimator. None if not enough data."""
    if EWMA_STATE['sigma_sq'] is None or EWMA_STATE['n_updates'] < 30:
        return None
    # σ is per-tick. Spot is polled every CFG['spot_poll_sec'] seconds.
    sigma_per_tick = math.sqrt(EWMA_STATE['sigma_sq'])
    ticks_per_year = (365.25 * 24 * 3600) / CFG['spot_poll_sec']
    return sigma_per_tick * math.sqrt(ticks_per_year)


# ─────────────────────────────────────────────────────────────────
# 2. Hampel filter for spot outlier rejection
#    Reject any spot tick that is > k × MAD away from the rolling median
#    Default k=3 (Hampel's identifier; mild outliers)
# ─────────────────────────────────────────────────────────────────

def _median(xs):
    s = sorted(xs); n = len(s)
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2


def _mad(xs, med):
    return _median([abs(x - med) for x in xs])


def spot_is_outlier(new_price: float, window: int = 30, k: float = 3.0,
                    min_abs_threshold: float = 50.0,
                    min_pct_threshold: float = 0.005,
                    max_stale_sec: float = 60.0) -> bool:
    """Hampel filter — True if new_price is an anomalous spike.

    Three guards prevent over-rejection:
      1. Absolute floor ($50) — for normal BTC ticks during quiet periods.
      2. Percent floor (0.5%) — at $80K spot, threshold ≥ $400 so a trending
         move ($300 in one tick) doesn't look like an outlier.
      3. Watchdog (60s) — if we haven't accepted a tick in too long, force-
         accept. Prevents the "median anchored to stale price" failure mode
         where the filter rejects everything forever after spot moves more
         than the threshold while the bot was running.
    """
    # Watchdog: if our last-accepted tick is stale, bypass the filter entirely.
    # The median is anchored to old data and can't recover until we let
    # something through.
    with _LOCK:
        last_ts = SPOT.get('ts')
    if last_ts is not None:
        age = (datetime.now(timezone.utc) - last_ts).total_seconds()
        if age > max_stale_sec:
            return False  # force-accept, will rebuild history from here
    with _LOCK:
        hist = [p for _, p in list(SPOT['history'])[-window:]]
    if len(hist) < 10:
        return False
    med = _median(hist)
    mad = _mad(hist, med)
    # MAD-to-σ scale: ~1.4826
    threshold = max(min_abs_threshold,
                    min_pct_threshold * med,
                    k * 1.4826 * mad)
    return abs(new_price - med) > threshold


# ─────────────────────────────────────────────────────────────────
# 3. Welford running stats — used by SPRT for win-rate
# ─────────────────────────────────────────────────────────────────

class Welford:
    """Online mean + variance, numerically stable."""
    __slots__ = ('n', 'mean', 'M2')
    def __init__(self):
        self.n = 0; self.mean = 0.0; self.M2 = 0.0
    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2
    def variance(self):
        return self.M2 / max(1, self.n - 1)
    def stddev(self):
        return math.sqrt(self.variance())


# ─────────────────────────────────────────────────────────────────
# 4. Black-Scholes binary with drift correction
#    Standard: d = (spot − K) / σ√t  (assumes zero drift)
#    With drift: d = (ln(spot/K) + μ × t) / (σ × √t × spot)
#
#    We measure μ from recent log-returns (annualized).
#    For 30-60min windows, BTC's daily drift is essentially noise so
#    we cap |μ| at ±0.5 annual (±50% / year) to prevent extreme estimates.
# ─────────────────────────────────────────────────────────────────

DRIFT_STATE = {'log_returns': [], 'last_price': None, 'window': 600}


def drift_update(new_price: float):
    if DRIFT_STATE['last_price'] is not None:
        r = math.log(new_price / DRIFT_STATE['last_price'])
        DRIFT_STATE['log_returns'].append(r)
        if len(DRIFT_STATE['log_returns']) > DRIFT_STATE['window']:
            DRIFT_STATE['log_returns'].pop(0)
    DRIFT_STATE['last_price'] = new_price


def estimated_drift_annual() -> float:
    """Return capped annualized drift. 0.0 if not enough data."""
    rs = DRIFT_STATE['log_returns']
    if len(rs) < 60:
        return 0.0
    mean_per_tick = sum(rs) / len(rs)
    ticks_per_year = (365.25 * 24 * 3600) / CFG['spot_poll_sec']
    drift = mean_per_tick * ticks_per_year
    return max(-0.5, min(0.5, drift))  # cap ±50%/yr


def fair_yes_with_drift(spot: float, strike: float, secs_to_close: float,
                       sigma_annual: float, mu_annual: float = 0.0) -> float:
    """Black-Scholes binary with drift. Returns P(spot_T > K)."""
    t = secs_to_close / (365.25 * 24 * 3600)
    if t <= 0 or sigma_annual <= 0 or spot <= 0:
        return 1.0 if spot > strike else 0.0
    # Risk-neutral N(d2) with drift μ
    sigma_sq = sigma_annual * sigma_annual
    d = (math.log(spot / strike) + (mu_annual - 0.5 * sigma_sq) * t) / (sigma_annual * math.sqrt(t))
    return _norm_cdf(d)


# ─────────────────────────────────────────────────────────────────
# 5. Merton jump-diffusion overlay for fat-tail fair value
#    fair_jump = Σ_{k=0}^{N} e^{-λt} × (λt)^k / k! × fair_BS(σ_k)
#    where σ_k incorporates k jumps. We approximate with N=5.
#    Conservative: increases P(extreme moves), making fair_yes lower
#    near strikes (i.e., MORE skeptical of deep-ITM contracts).
# ─────────────────────────────────────────────────────────────────

def fair_yes_jump(spot: float, strike: float, secs_to_close: float,
                  sigma_annual: float, mu_annual: float = 0.0,
                  jump_rate_per_year: float = 12.0,
                  jump_size_pct: float = 0.02) -> float:
    """Jump-diffusion fair value. jump_rate=12 means ~12 jumps/yr expected.
    Approximates BTC's ~monthly significant move pattern."""
    t = secs_to_close / (365.25 * 24 * 3600)
    if t <= 0 or sigma_annual <= 0:
        return 1.0 if spot > strike else 0.0

    lambda_t = jump_rate_per_year * t
    # Poisson probabilities for 0..5 jumps
    fair = 0.0
    fact = 1.0
    p_total = 0.0
    for k in range(6):
        if k > 0: fact *= k
        p_k = math.exp(-lambda_t) * (lambda_t ** k) / fact
        p_total += p_k
        # Effective σ inflated by k jumps
        sigma_k_sq = sigma_annual * sigma_annual + k * (jump_size_pct ** 2) / t
        sigma_k = math.sqrt(max(0.0001, sigma_k_sq))
        fair += p_k * fair_yes_with_drift(spot, strike, secs_to_close, sigma_k, mu_annual)
    # Normalize for truncation
    return fair / p_total


# ─────────────────────────────────────────────────────────────────
# 6. Kelly fraction for position sizing
#    f* = (p×b − q) / b   where p=win prob, q=1-p, b=win/loss ratio
#    For binary contracts at price P:
#       win = (1 - P - fee)
#       loss = (P + fee)
#       b = win / loss
#    We use fractional Kelly (1/4) to limit drawdowns.
# ─────────────────────────────────────────────────────────────────

def kelly_qty(win_prob: float, entry_price: float, bankroll: float,
              fraction: float = 0.25, max_qty: int = 100) -> int:
    """Returns Kelly-optimal qty, fractionalized + capped."""
    fee = kalshi_fee(entry_price)
    win = 1.0 - entry_price - fee
    loss = entry_price + fee
    if win <= 0 or loss <= 0:
        return 0
    b = win / loss
    p = win_prob
    q = 1.0 - p
    f_full = (p * b - q) / b
    if f_full <= 0:
        return 0
    f = f_full * fraction
    dollar_size = bankroll * f
    qty = int(dollar_size / entry_price)
    return min(max(qty, 1), max_qty)


# ─────────────────────────────────────────────────────────────────
# 7. SPRT (Sequential Probability Ratio Test) for auto-disable
#    H0: win rate = p_breakeven (strategy is breakeven)
#    H1: win rate = p_target (strategy has edge)
#    We track log-likelihood ratio after each trade.
#    If LLR drops below threshold A → DISABLE that tier (no edge)
#    If LLR rises above threshold B → CONFIRMED (keep running)
# ─────────────────────────────────────────────────────────────────

# SPRT state per tier (persists in-memory; resets on bot restart)
SPRT_STATE = {
    1: {'llr': 0.0, 'p_be': 0.50, 'p_target': 0.77, 'n': 0, 'disabled': False},
    2: {'llr': 0.0, 'p_be': 0.97, 'p_target': 0.985, 'n': 0, 'disabled': False},
    3: {'llr': 0.0, 'p_be': 0.77, 'p_target': 0.79, 'n': 0, 'disabled': False},
}
# Wald boundaries for α=0.05 (false-disable rate), β=0.10 (false-confirm rate)
SPRT_LOWER = math.log(0.10 / (1 - 0.05))   # ≈ −2.25  → disable
SPRT_UPPER = math.log((1 - 0.10) / 0.05)   # ≈ +2.89  → confirmed


def sprt_record_outcome(tier: int, won: bool):
    """Update LLR after a settled trade. Disable tier if LLR drops too low."""
    st = SPRT_STATE.get(tier)
    if st is None or st['disabled']:
        return
    p1, p0 = st['p_target'], st['p_be']
    if won:
        delta = math.log(p1 / p0)
    else:
        delta = math.log((1 - p1) / (1 - p0))
    st['llr'] += delta
    st['n'] += 1
    # Only adjudicate after at least 20 trades
    if st['n'] < 20:
        return
    if st['llr'] <= SPRT_LOWER:
        st['disabled'] = True
        _log(f'SPRT: Tier {tier} AUTO-DISABLED (n={st["n"]}, llr={st["llr"]:.2f}). '
             f'Win rate below {p0*100:.0f}%.')
    elif st['llr'] >= SPRT_UPPER:
        _log(f'SPRT: Tier {tier} CONFIRMED (n={st["n"]}, llr={st["llr"]:.2f})')
        # Don't actually act — just log confirmation


def sprt_active(tier: int) -> bool:
    """Returns True if tier should still trade (not auto-disabled)."""
    return not SPRT_STATE.get(tier, {}).get('disabled', False)


def sprt_state(tier: int) -> dict:
    return dict(SPRT_STATE.get(tier, {}))


# ─────────────────────────────────────────────────────────────────
# 8. Unified fair-value function — used by Tier 2 scanner
# ─────────────────────────────────────────────────────────────────

def fair_value_robust(spot: float, strike: float, secs_to_close: float) -> Optional[dict]:
    """Returns dict with multiple fair-value estimates + a consensus.
    Used by Tier 2 instead of the original simple BS calc."""
    # Get sigma from both sources, take the MAX (more conservative)
    sigma_realized = _causal_sigma()
    sigma_ewma = ewma_sigma_annual()
    if sigma_realized is None and sigma_ewma is None:
        return None
    sigma = max(sigma_realized or 0, sigma_ewma or 0)
    # Apply floor + padding (existing logic)
    sigma = max(sigma, CFG.get('t2_sigma_floor_annual', 0.35))
    sigma_padded = sigma * (1.0 + CFG.get('sigma_uncertainty_discount', 0.50))

    mu = estimated_drift_annual()

    fair_bs = fair_yes_with_drift(spot, strike, secs_to_close, sigma_padded, mu)
    fair_jump = fair_yes_jump(spot, strike, secs_to_close, sigma_padded, mu,
                              jump_rate_per_year=CFG.get('jump_rate', 12.0),
                              jump_size_pct=CFG.get('jump_size_pct', 0.02))

    # Consensus: take MIN (most conservative) for buying YES.
    # When spot > K (deep ITM YES), both estimates should be high; the lower
    # is the floor we trust.
    fair_consensus = min(fair_bs, fair_jump)

    return {
        'sigma_realized': sigma_realized,
        'sigma_ewma': sigma_ewma,
        'sigma_used': sigma_padded,
        'mu_annual': mu,
        'fair_bs': fair_bs,
        'fair_jump': fair_jump,
        'fair': fair_consensus,
    }


print('Math models ready: EWMA σ, drift correction, jump-diffusion,')
print('                   Hampel outlier filter, Kelly sizing, SPRT auto-disable.')


Math models ready: EWMA σ, drift correction, jump-diffusion,
                   Hampel outlier filter, Kelly sizing, SPRT auto-disable.


In [10]:
# § 3 — Signal Detection
#
# Tier 1 — Monotonicity arbitrage (RISK-FREE)
#   Pairs (mkt_lo, mkt_hi) on the same event where strike_hi > strike_lo
#   AND yes_bid_hi > yes_ask_lo + total_fees.
#   Trade: BUY YES(K_lo) at ask_lo, BUY NO(K_hi) at (1 - bid_hi).
#
# Tier 2 — Deep-ITM convergence
#   Black-Scholes binary fair value vs market price, deep-ITM only.
#   Filter: price >= 0.88, fair >= 0.94, edge >= 0.5c, TTC in [15min, 60min].


@dataclass
class T1Signal:
    pair_id: str
    mkt_lo: str
    mkt_hi: str
    strike_lo: float
    strike_hi: float
    ask_lo: float
    bid_hi: float
    qty: int
    net_edge_cents: float


@dataclass
class T2Signal:
    ticker: str
    side: str            # 'yes' or 'no'
    strike: float
    price: float
    fair_value: float
    edge_cents: float
    qty: int
    secs_remaining: float
    sigma: float


@dataclass
class T3Signal:
    """OTM persistence NO buy — buy NO when spot has been < strike for ≥5min."""
    ticker: str
    strike: float
    no_price: float
    yes_bid: float
    qty: int
    secs_remaining: float
    persistence_min: float
    spot_dist: float


def _spot_was_below(strike: float, secs_ago: float) -> Optional[bool]:
    """Was spot < strike `secs_ago` seconds ago? None if no history."""
    target = datetime.now(timezone.utc) - timedelta(seconds=secs_ago)
    with _LOCK:
        hist = list(SPOT['history'])
    if not hist:
        return None
    for ts, price in reversed(hist):
        if ts <= target:
            return price < strike
    return None  # not enough history


def _spot_move_over(secs: float) -> Optional[float]:
    """|spot(now) - spot(now - secs)|. None if not enough history."""
    target = datetime.now(timezone.utc) - timedelta(seconds=secs)
    with _LOCK:
        hist = list(SPOT['history'])
        now_price = SPOT.get('price')
    if not hist or now_price is None:
        return None
    for ts, price in reversed(hist):
        if ts <= target:
            return abs(now_price - price)
    return None  # not enough history yet


def scan_t1_monotonicity():
    if not CFG['t1_enabled']:
        return []
    try:
        if not sprt_active(1):
            return []
    except NameError:
        pass
    # V5 calm-regime filter — only trade T1 when spot has been stable.
    # Volatile spot ⇒ HFT activity ⇒ arbs evaporate before we can fill both legs.
    if CFG.get('t1_calm_filter_enabled', False):
        move30 = _spot_move_over(30.0)
        if move30 is None or move30 > CFG['t1_calm_max_spot_move_30s']:
            return []
    with _LOCK:
        snap = {k: dict(v) for k, v in BOOKS.items()}

    rows = []
    for tk, b in snap.items():
        if b.get('status') != 'active' or b.get('floor') is None:
            continue
        ya = b.get('yes_ask'); yb = b.get('yes_bid')
        if ya is None or yb is None or yb <= 0.01 or ya >= 0.99:
            continue
        ya_qty = b.get('yes_ask_qty') or 0
        yb_qty = b.get('yes_bid_qty') or 0
        rows.append({'tk': tk, 'K': float(b['floor']),
                     'ya': ya, 'yb': yb, 'ya_qty': ya_qty, 'yb_qty': yb_qty})
    rows.sort(key=lambda r: r['K'])

    opps = []
    for i, lo in enumerate(rows):
        for hi in rows[i+1:]:
            if hi['yb'] <= lo['ya']:
                continue
            gross = hi['yb'] - lo['ya']
            fees = kalshi_fee(lo['ya']) + kalshi_fee(1.0 - hi['yb'])
            net = gross - fees
            if net < CFG['t1_min_net_edge_cents'] / 100:
                continue
            qty = min(lo['ya_qty'] or 1, hi['yb_qty'] or 1,
                      CFG['t1_max_qty_per_leg'])
            cost_per_unit = lo['ya'] + (1.0 - hi['yb'])
            if cost_per_unit > 0:
                qty = min(qty, max(1, int(CFG['t1_max_dollars_per_pair'] / cost_per_unit)))
            if qty < 1:
                continue
            opps.append(T1Signal(
                pair_id=f't1-{uuid.uuid4().hex[:10]}',
                mkt_lo=lo['tk'], mkt_hi=hi['tk'],
                strike_lo=lo['K'], strike_hi=hi['K'],
                ask_lo=lo['ya'], bid_hi=hi['yb'],
                qty=int(qty), net_edge_cents=net * 100,
            ))
    opps.sort(key=lambda o: o.net_edge_cents, reverse=True)
    return opps


def scan_t2_deep_itm():
    if not CFG['t2_enabled']:
        return []
    # SPRT auto-disable check (if math models are loaded)
    try:
        if not sprt_active(2):
            return []
    except NameError:
        pass
    with _LOCK:
        spot = SPOT.get('price')
        close_time = TRACKED.get('close_time')
        snap = {k: dict(v) for k, v in BOOKS.items()}
    if spot is None or close_time is None:
        return []
    now = datetime.now(timezone.utc)
    secs = (close_time - now).total_seconds()
    if not (CFG['t2_min_secs_to_close'] <= secs <= CFG['t2_max_secs_to_close']):
        return []

    # Hard minimum distance from strike — defense against the "spot blew through"
    # failure mode where model sigma underestimates a regime shift.
    min_dist = max(CFG['t2_min_dollar_distance_from_strike'],
                   spot * CFG['t2_min_pct_distance_from_strike'])

    max_spread = CFG.get('t2_max_spread_at_entry', 0.05)

    signals = []
    for tk, b in snap.items():
        if b.get('status') != 'active' or b.get('floor') is None:
            continue
        K = float(b['floor'])
        if abs(spot - K) < min_dist:
            continue
        ya = b.get('yes_ask'); yb = b.get('yes_bid')
        ya_qty = b.get('yes_ask_qty') or 1
        yb_qty = b.get('yes_bid_qty') or 1
        if ya is not None and yb is not None and (ya - yb) > max_spread:
            continue

        # Use robust fair value (EWMA σ + drift + jump-diffusion consensus)
        # Falls back to plain Black-Scholes if math models not loaded.
        try:
            fv = fair_value_robust(spot, K, secs)
            if fv is None:
                continue
            fair_yes = fv['fair']
            sigma = fv['sigma_used']
        except NameError:
            sigma = max(_causal_sigma() or 0, CFG['t2_sigma_floor_annual'])
            sigma_c = sigma * (1.0 + CFG['sigma_uncertainty_discount'])
            sig_s = sigma_c / math.sqrt(365.25 * 24 * 3600)
            sig_rem = sig_s * spot * math.sqrt(secs)
            if sig_rem <= 0:
                continue
            d = abs(spot - K) / sig_rem
            fair_yes = _norm_cdf(d) if spot > K else 1.0 - _norm_cdf(d)
        fair_no = 1.0 - fair_yes

        # YES-side buy
        if ya is not None and CFG['t2_min_price'] <= ya <= 0.97 and fair_yes >= CFG['t2_min_fair']:
            edge = fair_yes - ya - kalshi_fee(ya)
            if edge >= CFG['t2_min_edge_cents'] / 100:
                qty = min(ya_qty, CFG['t2_max_qty_per_strike'],
                          max(1, int(CFG['t2_max_dollars_per_trade'] / max(0.01, ya))))
                signals.append(T2Signal(
                    ticker=tk, side='yes', strike=K, price=ya, fair_value=fair_yes,
                    edge_cents=edge * 100, qty=int(qty),
                    secs_remaining=secs, sigma=sigma))

        # NO-side buy
        if yb is not None and CFG['t2_min_price'] <= (1.0 - yb) <= 0.97 and fair_no >= CFG['t2_min_fair']:
            no_price = 1.0 - yb
            edge = fair_no - no_price - kalshi_fee(no_price)
            if edge >= CFG['t2_min_edge_cents'] / 100:
                qty = min(yb_qty, CFG['t2_max_qty_per_strike'],
                          max(1, int(CFG['t2_max_dollars_per_trade'] / max(0.01, no_price))))
                signals.append(T2Signal(
                    ticker=tk, side='no', strike=K, price=no_price, fair_value=fair_no,
                    edge_cents=edge * 100, qty=int(qty),
                    secs_remaining=secs, sigma=sigma))

    signals.sort(key=lambda s: s.edge_cents, reverse=True)
    return signals


def scan_t3_otm_persistence():
    """Tier 3 — buy NO when spot has been < strike for ≥5 min."""
    if not CFG.get('t3_enabled', False):
        return []
    try:
        if not sprt_active(3):
            return []
    except NameError:
        pass
    with _LOCK:
        spot = SPOT.get('price')
        close_time = TRACKED.get('close_time')
        snap = {k: dict(v) for k, v in BOOKS.items()}
    if spot is None or close_time is None:
        return []
    now = datetime.now(timezone.utc)
    secs = (close_time - now).total_seconds()
    if not (CFG['t3_min_secs_to_close'] <= secs <= CFG['t3_max_secs_to_close']):
        return []

    persistence_s = CFG['t3_persistence_seconds']
    # Need at least 5 min of history to verify
    with _LOCK:
        hist = list(SPOT['history'])
    if len(hist) < 30:  # ~60s with 2s polling
        return []

    signals = []
    for tk, b in snap.items():
        if b.get('status') != 'active' or b.get('floor') is None:
            continue
        K = float(b['floor'])
        if spot >= K:
            continue  # need OTM for YES (spot < K)
        dist = K - spot
        if dist < CFG['t3_min_strike_distance']:
            continue
        yb = b.get('yes_bid')
        if yb is None:
            continue
        if not (CFG['t3_min_yes_bid'] <= yb <= CFG['t3_max_yes_bid']):
            continue

        # Persistence check: spot below K for full persistence window
        was_below_300 = _spot_was_below(K, persistence_s)
        was_below_180 = _spot_was_below(K, persistence_s * 0.6)
        was_below_60 = _spot_was_below(K, persistence_s * 0.2)
        if was_below_300 is None or not (was_below_60 and was_below_180 and was_below_300):
            continue

        no_price = 1.0 - yb
        yb_qty = b.get('yes_bid_qty') or 1
        qty = min(yb_qty, CFG['t3_max_qty_per_strike'],
                  max(1, int(CFG['t3_max_dollars_per_trade'] / max(0.01, no_price))))
        signals.append(T3Signal(
            ticker=tk, strike=K, no_price=no_price, yes_bid=yb,
            qty=int(qty), secs_remaining=secs,
            persistence_min=persistence_s / 60.0, spot_dist=dist))

    signals.sort(key=lambda s: s.spot_dist, reverse=True)  # prefer far OTM
    return signals


def scan_all_signals():
    t1 = scan_t1_monotonicity()
    t2 = scan_t2_deep_itm()
    t3 = scan_t3_otm_persistence()
    blocked = set()
    for o in t1:
        blocked.add(o.mkt_lo); blocked.add(o.mkt_hi)
    t2_clean = [s for s in t2 if s.ticker not in blocked]
    t3_clean = [s for s in t3 if s.ticker not in blocked]
    return {'tier1': t1, 'tier2': t2_clean, 'tier3': t3_clean}


print('Signal detection ready (Tier 1 + Tier 2 + Tier 3).')


Signal detection ready (Tier 1 + Tier 2 + Tier 3).


In [11]:
# § 4 — Execution: Orders, Positions, Settlement
#
# Tracks live signal-flow telemetry so the dashboard can show what's being
# scanned vs. executed vs. skipped per cycle.

from collections import deque

SIGNAL_FLOW = {
    'cycles': 0,
    'totals': {'t1_seen': 0, 't2_seen': 0, 't3_seen': 0,
               't1_exec': 0, 't2_exec': 0, 't3_exec': 0},
    'last_cycle': {'t1_seen': 0, 't2_seen': 0, 't3_seen': 0,
                   't1_exec': 0, 't2_exec': 0, 't3_exec': 0},
    'skip_reasons': {},   # reason -> count
    'recent': deque(maxlen=20),  # tuples of (ts_str, tier, ticker, status, detail)
}


def _flow_record(tier: int, ticker: str, status: str, detail: str = ''):
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S')
    SIGNAL_FLOW['recent'].append((ts, tier, ticker, status, detail))
    if status.startswith('skip:'):
        reason = status[5:]
        SIGNAL_FLOW['skip_reasons'][reason] = SIGNAL_FLOW['skip_reasons'].get(reason, 0) + 1

TRADES_INIT_SQL = (
    "CREATE TABLE IF NOT EXISTS trades ("
    "id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT NOT NULL,"
    " tier INTEGER NOT NULL, pair_id TEXT,"
    " event_ticker TEXT, market_ticker TEXT NOT NULL,"
    " side TEXT NOT NULL, action TEXT NOT NULL,"
    " contracts INTEGER NOT NULL, entry_price REAL NOT NULL,"
    " edge_cents REAL, fair_value REAL, spot REAL, sigma REAL,"
    " secs_remaining REAL, mode TEXT DEFAULT 'paper',"
    " order_id TEXT, client_order_id TEXT,"
    " settled INTEGER DEFAULT 0, settlement_result TEXT,"
    " pnl REAL, settled_at TEXT);"
    "CREATE INDEX IF NOT EXISTS idx_trades_settled ON trades(settled);"
    "CREATE INDEX IF NOT EXISTS idx_trades_market ON trades(market_ticker, settled);"
    "CREATE INDEX IF NOT EXISTS idx_trades_pair ON trades(pair_id);"
)


def _trades_conn():
    conn = sqlite3.connect(CFG['trades_db_path'])
    conn.row_factory = sqlite3.Row
    return conn


def _init_trades_db():
    conn = _trades_conn()
    conn.executescript(TRADES_INIT_SQL)
    conn.commit(); conn.close()

_init_trades_db()


def _record_trade(tier, market_ticker, side, contracts, entry_price,
                  edge_cents=None, fair_value=None, spot=None, sigma=None,
                  secs_remaining=None, pair_id=None, order_id=None,
                  client_order_id=None, event_ticker=None):
    conn = _trades_conn()
    cur = conn.execute(
        'INSERT INTO trades(ts, tier, pair_id, event_ticker, market_ticker, side, action,'
        ' contracts, entry_price, edge_cents, fair_value, spot, sigma, secs_remaining,'
        ' mode, order_id, client_order_id)'
        ' VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
        (datetime.now(timezone.utc).isoformat(), tier, pair_id,
         event_ticker or TRACKED.get('event'), market_ticker, side, 'buy',
         contracts, entry_price, edge_cents, fair_value, spot, sigma, secs_remaining,
         CFG['mode'], order_id, client_order_id))
    trade_id = cur.lastrowid
    conn.commit(); conn.close()
    return trade_id


def _open_positions():
    conn = _trades_conn()
    rows = [dict(r) for r in conn.execute('SELECT * FROM trades WHERE settled = 0').fetchall()]
    conn.close()
    return rows


def _market_is_open(ticker):
    conn = _trades_conn()
    n = conn.execute('SELECT COUNT(*) FROM trades WHERE settled=0 AND market_ticker=?',
                     (ticker,)).fetchone()[0]
    conn.close()
    return n > 0


def _open_exposure():
    conn = _trades_conn()
    row = conn.execute('SELECT COALESCE(SUM(entry_price * contracts), 0) FROM trades WHERE settled=0').fetchone()
    conn.close()
    return float(row[0])


def _today_pnl():
    today = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0).isoformat()
    conn = _trades_conn()
    row = conn.execute('SELECT COALESCE(SUM(pnl), 0) FROM trades WHERE settled=1 AND settled_at >= ?',
                       (today,)).fetchone()
    conn.close()
    return float(row[0])


def _risk_preflight(market_tickers, total_cost):
    if _today_pnl() < CFG['daily_loss_limit']:
        return False, f'daily loss limit hit (${_today_pnl():.2f})'
    for tk in market_tickers:
        if _market_is_open(tk):
            return False, f'already open on {tk}'
    open_pos = _open_positions()
    if len(open_pos) >= CFG['max_concurrent_positions']:
        return False, f'open positions ({len(open_pos)}) >= cap'
    exp = _open_exposure()
    if exp + total_cost > CFG['max_total_exposure']:
        return False, f'exposure ${exp+total_cost:.0f} > cap ${CFG["max_total_exposure"]}'
    # Paper-account cash check (if paper mode + paper module loaded)
    try:
        if CFG['mode'] == 'paper' and not paper_has_cash(total_cost):
            return False, f'paper cash ${PAPER_ACCOUNT["cash"]:.2f} < cost ${total_cost:.2f}'
    except NameError:
        pass
    return True, ''


def _place_one(market_ticker, side, qty, limit_price):
    # Returns (order_id, client_order_id) or (None, None)
    if CFG['mode'] != 'live' or kalshi_live is None:
        return 'paper', f'paper-{uuid.uuid4().hex[:12]}'
    buf = CFG['order_buffer_cents'] / 100.0
    p = min(0.99, limit_price + buf)
    cents = int(round(p * 100))
    resp = kalshi_live.place_order(
        ticker=market_ticker, side=side, action='buy',
        count=qty, yes_price_cents=cents,
        expiration_sec=CFG['order_expiration_sec'])
    if 'error' in resp:
        _log(f'ORDER FAIL {resp.get("status","")}: {resp["error"][:120]}')
        return None, None
    o = resp.get('order', {})
    return o.get('order_id'), o.get('client_order_id')


def execute_t1(sig):
    cost_per_unit = sig.ask_lo + (1.0 - sig.bid_hi)
    total_cost = cost_per_unit * sig.qty
    ok, why = _risk_preflight([sig.mkt_lo, sig.mkt_hi], total_cost)
    if not ok:
        _log(f'T1 SKIP {sig.mkt_lo}/{sig.mkt_hi}: {why}')
        _flow_record(1, f'{sig.mkt_lo[-12:]}↔{sig.mkt_hi[-12:]}', f'skip:{why}',
                     f'edge={sig.net_edge_cents:.1f}c qty={sig.qty}')
        return False

    oid_a, cid_a = _place_one(sig.mkt_lo, 'yes', sig.qty, sig.ask_lo)
    if oid_a is None:
        _flow_record(1, sig.mkt_lo[-12:], 'skip:order_failed_leg1', '')
        return False
    _record_trade(tier=1, market_ticker=sig.mkt_lo, side='yes',
                  contracts=sig.qty, entry_price=sig.ask_lo,
                  edge_cents=sig.net_edge_cents, pair_id=sig.pair_id,
                  order_id=oid_a, client_order_id=cid_a)

    oid_b, cid_b = _place_one(sig.mkt_hi, 'no', sig.qty, 1.0 - sig.bid_hi)
    if oid_b is None:
        _log(f'T1 PARTIAL: leg1 filled, leg2 failed. MANUAL UNWIND on {sig.mkt_lo}')
        _flow_record(1, sig.mkt_hi[-12:], 'skip:order_failed_leg2',
                     'MANUAL UNWIND REQUIRED')
        return False
    _record_trade(tier=1, market_ticker=sig.mkt_hi, side='no',
                  contracts=sig.qty, entry_price=1.0 - sig.bid_hi,
                  edge_cents=sig.net_edge_cents, pair_id=sig.pair_id,
                  order_id=oid_b, client_order_id=cid_b)

    try: paper_open_trade(total_cost)
    except NameError: pass
    BOT_STATE['trades_this_session'] += 1
    SIGNAL_FLOW['totals']['t1_exec'] += 1
    SIGNAL_FLOW['last_cycle']['t1_exec'] += 1
    _flow_record(1, f'{sig.mkt_lo[-12:]}+{sig.mkt_hi[-12:]}', 'EXEC',
                 f'edge={sig.net_edge_cents:.1f}c qty={sig.qty}')
    _log(f'T1 PAIR {sig.pair_id[-6:]}: '
         f'BUY {sig.mkt_lo} yes x{sig.qty}@{sig.ask_lo:.2f} + '
         f'BUY {sig.mkt_hi} no x{sig.qty}@{(1-sig.bid_hi):.2f}  edge={sig.net_edge_cents:.1f}c')
    return True


def execute_t2(sig):
    cost = sig.price * sig.qty
    ok, why = _risk_preflight([sig.ticker], cost)
    if not ok:
        _log(f'T2 SKIP {sig.ticker}: {why}')
        _flow_record(2, sig.ticker[-16:], f'skip:{why}',
                     f'{sig.side} @${sig.price:.2f} edge={sig.edge_cents:.1f}c')
        return False
    oid, cid = _place_one(sig.ticker, sig.side, sig.qty, sig.price)
    if oid is None:
        _flow_record(2, sig.ticker[-16:], 'skip:order_failed', '')
        return False
    _record_trade(tier=2, market_ticker=sig.ticker, side=sig.side,
                  contracts=sig.qty, entry_price=sig.price,
                  edge_cents=sig.edge_cents, fair_value=sig.fair_value,
                  spot=SPOT.get('price'), sigma=sig.sigma,
                  secs_remaining=sig.secs_remaining,
                  order_id=oid, client_order_id=cid)
    try: paper_open_trade(cost)
    except NameError: pass
    BOT_STATE['trades_this_session'] += 1
    SIGNAL_FLOW['totals']['t2_exec'] += 1
    SIGNAL_FLOW['last_cycle']['t2_exec'] += 1
    _flow_record(2, sig.ticker[-16:], 'EXEC',
                 f'{sig.side} x{sig.qty} @${sig.price:.2f} edge={sig.edge_cents:.1f}c')
    _log(f'T2 ENTRY: {sig.ticker} {sig.side} x{sig.qty} @ ${sig.price:.2f}  '
         f'fair={sig.fair_value:.3f}  edge={sig.edge_cents:.1f}c')
    return True


def execute_t3(sig):
    cost = sig.no_price * sig.qty
    ok, why = _risk_preflight([sig.ticker], cost)
    if not ok:
        _log(f'T3 SKIP {sig.ticker}: {why}')
        _flow_record(3, sig.ticker[-16:], f'skip:{why}',
                     f'no @${sig.no_price:.2f} dist=${sig.spot_dist:.0f}')
        return False
    oid, cid = _place_one(sig.ticker, 'no', sig.qty, sig.no_price)
    if oid is None:
        _flow_record(3, sig.ticker[-16:], 'skip:order_failed', '')
        return False
    _record_trade(tier=3, market_ticker=sig.ticker, side='no',
                  contracts=sig.qty, entry_price=sig.no_price,
                  spot=SPOT.get('price'),
                  secs_remaining=sig.secs_remaining,
                  order_id=oid, client_order_id=cid)
    try: paper_open_trade(cost)
    except NameError: pass
    BOT_STATE['trades_this_session'] += 1
    SIGNAL_FLOW['totals']['t3_exec'] += 1
    SIGNAL_FLOW['last_cycle']['t3_exec'] += 1
    _flow_record(3, sig.ticker[-16:], 'EXEC',
                 f'no x{sig.qty} @${sig.no_price:.2f} dist=${sig.spot_dist:.0f}')
    _log(f'T3 ENTRY: {sig.ticker} no x{sig.qty} @ ${sig.no_price:.2f}  '
         f'dist=${sig.spot_dist:.0f}  persist={sig.persistence_min:.0f}min')
    return True


def check_settlements():
    for pos in _open_positions():
        tk = pos['market_ticker']
        try:
            m = kalshi_prod.get_market(tk).get('market', {})
            status = (m.get('status') or '').lower()
            if status not in ('settled', 'finalized', 'closed', 'determined'):
                continue
            result = (m.get('result') or '').lower()
            if result not in ('yes', 'no'):
                continue
            payout = 1.0 if pos['side'] == result else 0.0
            pnl_per_c = payout - pos['entry_price'] - kalshi_fee(pos['entry_price'])
            pnl = pnl_per_c * pos['contracts']
            conn = _trades_conn()
            conn.execute('UPDATE trades SET settled=1, settlement_result=?, pnl=?, settled_at=? WHERE id=?',
                         (result, pnl, datetime.now(timezone.utc).isoformat(), pos['id']))
            conn.commit(); conn.close()
            tag = f'T{pos["tier"]}'
            won = pnl > 0
            # Feed SPRT (if loaded)
            try: sprt_record_outcome(pos['tier'], won)
            except NameError: pass
            # Update paper account (if loaded + we're in paper mode)
            try:
                if CFG['mode'] == 'paper':
                    paper_settle_trade(pos['contracts'], pos['entry_price'], pnl)
            except NameError: pass
            _log(f'{tag} SETTLE: {tk} {pos["side"]} -> {result}  pnl=${pnl:+.2f}')
        except Exception as e:
            _log(f'settle err {tk}: {e}')


print('Execution engine ready.')


Execution engine ready.


In [12]:
# § 4b — $100 Paper Trading Account
#
# Simulates a real broker account in paper mode:
#   - Tracks cash, locked (in open positions), realized PnL
#   - Risk preflight checks `paper_has_cash(cost)` before approving trades
#   - On settlement: deduct fees, credit payout, release locked cash
#   - Persists state in output/paper_account.db so restart doesn't reset
#
# Live mode is unaffected — paper account is only consulted when CFG['mode'] == 'paper'

PAPER_DB_PATH = 'output/paper_account.db'
PAPER_STARTING_CASH = 100.00  # ← change here for different account size

PAPER_ACCOUNT = {
    'starting_cash': PAPER_STARTING_CASH,
    'cash': PAPER_STARTING_CASH,
    'locked': 0.0,
    'realized_pnl': 0.0,
    'trades_opened': 0,
    'trades_settled': 0,
    'wins': 0,
    'losses': 0,
    'peak_equity': PAPER_STARTING_CASH,
    'max_drawdown': 0.0,  # max % from peak
    'session_started_at': None,
}


def _paper_db():
    conn = sqlite3.connect(PAPER_DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def _init_paper_db():
    conn = _paper_db()
    conn.executescript('''
    CREATE TABLE IF NOT EXISTS paper_state (
        key TEXT PRIMARY KEY, value REAL
    );
    CREATE TABLE IF NOT EXISTS paper_events (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        ts TEXT NOT NULL,
        event_type TEXT NOT NULL,
        amount REAL,
        cash_after REAL,
        locked_after REAL,
        note TEXT
    );
    ''')
    conn.commit(); conn.close()


def _save_paper_state():
    conn = _paper_db()
    for k, v in PAPER_ACCOUNT.items():
        if isinstance(v, (int, float)):
            conn.execute('INSERT OR REPLACE INTO paper_state(key, value) VALUES(?, ?)',
                         (k, float(v)))
    conn.commit(); conn.close()


def _load_paper_state():
    conn = _paper_db()
    rows = conn.execute('SELECT key, value FROM paper_state').fetchall()
    conn.close()
    if not rows:
        return False
    for k, v in rows:
        if k in PAPER_ACCOUNT and isinstance(PAPER_ACCOUNT[k], (int, float)):
            PAPER_ACCOUNT[k] = v
    return True


def _log_paper_event(event_type: str, amount: float, note: str = ''):
    conn = _paper_db()
    conn.execute(
        'INSERT INTO paper_events(ts, event_type, amount, cash_after, locked_after, note) '
        'VALUES(?,?,?,?,?,?)',
        (datetime.now(timezone.utc).isoformat(), event_type, amount,
         PAPER_ACCOUNT['cash'], PAPER_ACCOUNT['locked'], note))
    conn.commit(); conn.close()


def paper_reset(starting_cash: float = None):
    """Reset paper account to fresh state. Use when you want a clean start."""
    if starting_cash is None:
        starting_cash = PAPER_STARTING_CASH
    PAPER_ACCOUNT.update({
        'starting_cash': starting_cash,
        'cash': starting_cash,
        'locked': 0.0,
        'realized_pnl': 0.0,
        'trades_opened': 0,
        'trades_settled': 0,
        'wins': 0,
        'losses': 0,
        'peak_equity': starting_cash,
        'max_drawdown': 0.0,
        'session_started_at': datetime.now(timezone.utc).isoformat(),
    })
    # Wipe history
    conn = _paper_db()
    conn.execute('DELETE FROM paper_state')
    conn.execute('DELETE FROM paper_events')
    conn.commit(); conn.close()
    _save_paper_state()
    _log_paper_event('reset', starting_cash, f'Reset to ${starting_cash}')
    print(f'Paper account reset: ${starting_cash:.2f} starting cash')


def paper_has_cash(cost: float) -> bool:
    """Risk preflight: can we afford this trade?"""
    return PAPER_ACCOUNT['cash'] >= cost


def paper_open_trade(cost: float):
    """Deduct cost from cash, add to locked. Called when a trade is opened."""
    PAPER_ACCOUNT['cash'] -= cost
    PAPER_ACCOUNT['locked'] += cost
    PAPER_ACCOUNT['trades_opened'] += 1
    _log_paper_event('open', -cost, f'cost ${cost:.2f}')
    _save_paper_state()


def paper_settle_trade(contracts: int, entry_price: float, pnl: float):
    """Settle a trade. pnl is the net PnL (already includes fees).
    The original cost was entry_price × contracts."""
    cost = entry_price * contracts
    PAPER_ACCOUNT['locked'] -= cost
    PAPER_ACCOUNT['cash'] += cost + pnl  # release cost + add PnL
    PAPER_ACCOUNT['realized_pnl'] += pnl
    PAPER_ACCOUNT['trades_settled'] += 1
    if pnl > 0:
        PAPER_ACCOUNT['wins'] += 1
    else:
        PAPER_ACCOUNT['losses'] += 1
    # Track peak equity + max drawdown
    equity = paper_equity_value()
    if equity > PAPER_ACCOUNT['peak_equity']:
        PAPER_ACCOUNT['peak_equity'] = equity
    dd = (PAPER_ACCOUNT['peak_equity'] - equity) / PAPER_ACCOUNT['peak_equity']
    if dd > PAPER_ACCOUNT['max_drawdown']:
        PAPER_ACCOUNT['max_drawdown'] = dd
    _log_paper_event('settle', pnl, f'pnl ${pnl:+.2f}')
    _save_paper_state()


def paper_equity_value() -> float:
    """Total account value = cash + locked (cost basis of open positions)."""
    return PAPER_ACCOUNT['cash'] + PAPER_ACCOUNT['locked']


def paper_mark_to_market() -> float:
    """Equity including unrealized PnL from open positions."""
    unrealized = 0.0
    for p in _open_positions():
        try:
            _, u_per_c, _ = _market_to_market(p)
            if u_per_c is not None:
                unrealized += u_per_c * p['contracts']
        except Exception:
            pass
    return paper_equity_value() + unrealized


def paper_stats():
    """One-shot snapshot of paper account performance."""
    starting = PAPER_ACCOUNT['starting_cash']
    equity = paper_equity_value()
    mtm = paper_mark_to_market()
    realized = PAPER_ACCOUNT['realized_pnl']
    unrealized = mtm - equity
    settled = PAPER_ACCOUNT['trades_settled']
    win_pct = (PAPER_ACCOUNT['wins'] / settled * 100) if settled else 0
    pct_return = (mtm - starting) / starting * 100
    print(f'╔══════════════════════════════════════════════════════════════════╗')
    print(f'║  PAPER ACCOUNT  —  starting ${starting:.2f}                                  ║')
    print(f'╠══════════════════════════════════════════════════════════════════╣')
    print(f'║  Cash available:       ${PAPER_ACCOUNT["cash"]:>10.2f}                          ║')
    print(f'║  Locked in positions:  ${PAPER_ACCOUNT["locked"]:>10.2f}                          ║')
    print(f'║  Cost-basis equity:    ${equity:>10.2f}                          ║')
    print(f'║  Mark-to-market:       ${mtm:>10.2f}    (unreal ${unrealized:+.2f})  ║')
    print(f'║  Realized PnL:         ${realized:>+10.2f}                          ║')
    print(f'║  Total return:         {pct_return:>+9.2f}%                           ║')
    print(f'║  Trades: opened={PAPER_ACCOUNT["trades_opened"]:>3}  settled={settled:>3}  '
          f'wins={PAPER_ACCOUNT["wins"]:>3}  losses={PAPER_ACCOUNT["losses"]:>3}  ║')
    if settled:
        print(f'║  Win rate:             {win_pct:>9.1f}%                            ║')
    print(f'║  Peak equity:          ${PAPER_ACCOUNT["peak_equity"]:>10.2f}                          ║')
    print(f'║  Max drawdown:         {PAPER_ACCOUNT["max_drawdown"]*100:>9.2f}%                           ║')
    print(f'╚══════════════════════════════════════════════════════════════════╝')


# Initialize on cell run
_init_paper_db()
if not _load_paper_state():
    paper_reset(PAPER_STARTING_CASH)
    print(f'Paper account initialized with ${PAPER_STARTING_CASH:.2f}.')
else:
    print(f'Paper account loaded: ${PAPER_ACCOUNT["cash"]:.2f} cash, '
          f'${PAPER_ACCOUNT["locked"]:.2f} locked, '
          f'realized PnL ${PAPER_ACCOUNT["realized_pnl"]:+.2f}')
    print(f'  → paper_reset() to start over.  paper_stats() for full snapshot.')


Paper account loaded: $100.27 cash, $0.88 locked, realized PnL $+1.15
  → paper_reset() to start over.  paper_stats() for full snapshot.


In [13]:
# § 5 — Main Loop

def _decision_loop():
    while BOT_STATE['running']:
        BOT_STATE['iter'] += 1
        SIGNAL_FLOW['cycles'] += 1
        # Reset per-cycle stats
        for k in SIGNAL_FLOW['last_cycle']:
            SIGNAL_FLOW['last_cycle'][k] = 0
        try:
            if BOT_STATE['iter'] % 12 == 0:
                check_settlements()

            sigs = scan_all_signals()
            n1, n2, n3 = len(sigs['tier1']), len(sigs['tier2']), len(sigs['tier3'])
            SIGNAL_FLOW['last_cycle']['t1_seen'] = n1
            SIGNAL_FLOW['last_cycle']['t2_seen'] = n2
            SIGNAL_FLOW['last_cycle']['t3_seen'] = n3
            SIGNAL_FLOW['totals']['t1_seen'] += n1
            SIGNAL_FLOW['totals']['t2_seen'] += n2
            SIGNAL_FLOW['totals']['t3_seen'] += n3

            # Tier 1 first (risk-free) — execute up to 1 per cycle
            for s in sigs['tier1']:
                if execute_t1(s):
                    break

            # Tier 2 — best remaining
            for s in sigs['tier2']:
                if execute_t2(s):
                    break

            # Tier 3 — best remaining
            for s in sigs['tier3']:
                if execute_t3(s):
                    break

        except Exception as e:
            _log(f'decision err: {e}')
        _sleep(CFG['decision_interval_sec'])


def start_bot():
    if BOT_STATE['running']:
        print('Bot already running.'); return
    BOT_STATE['running'] = True
    BOT_STATE['iter'] = 0
    BOT_STATE['log'] = []
    BOT_STATE['trades_this_session'] = 0
    _WS_STATE['msg_count'] = 0
    _WS_STATE['reconnect_count'] = 0
    _WS_STATE['subscribed_event'] = None
    _WS_STATE['last_msg_ts'] = None

    # Sync-bootstrap: find an event NOW so WS subscribes on first connect
    print('Bootstrapping: looking for active event...')
    try:
        be, bc, n = _scan_for_event()
        if be:
            _apply_tracked_event(be, bc)
            print(f'  → tracking {be} closes {bc} (from {n} candidates)')
        else:
            print(f'  → no events in 2min–2hr window (checked {n} candidates). '
                  f'event_tracker will retry every 60s.')
    except Exception as e:
        print(f'  bootstrap err: {e}')

    workers = [
        ('spot_poller', _spot_poller),
        ('ws_listener', _ws_listener),
        ('event_tracker', _event_tracker),
        ('tick_flusher', _tick_flusher),
        ('decision_loop', _decision_loop),
    ]
    threads = []
    for name, fn in workers:
        t = threading.Thread(target=fn, name=name, daemon=True)
        t.start()
        threads.append(t)
    BOT_STATE['threads'] = threads

    print(f'Bot started.  Mode={CFG["mode"]}  T1={CFG["t1_enabled"]}  T2={CFG["t2_enabled"]}')
    print(f'  Tier1: min_edge={CFG["t1_min_net_edge_cents"]}c max_qty={CFG["t1_max_qty_per_leg"]}')
    print(f'  Tier2: price>={CFG["t2_min_price"]} fair>={CFG["t2_min_fair"]} '
          f'ttc=[{CFG["t2_min_secs_to_close"]},{CFG["t2_max_secs_to_close"]}]s')
    print(f'  Risk: max_pos={CFG["max_concurrent_positions"]}  '
          f'exp_cap=${CFG["max_total_exposure"]}  daily_loss=${CFG["daily_loss_limit"]}')
    print(f'  status()  diagnostics()  tick_stats()  stop_bot()')


def stop_bot():
    BOT_STATE['running'] = False
    for t in BOT_STATE.get('threads', []):
        t.join(timeout=5)
    BOT_STATE['threads'] = []
    _WS_STATE['connected'] = False
    print(f'Bot stopped. Trades this session: {BOT_STATE["trades_this_session"]}')


print('Main loop ready.')


Main loop ready.


In [14]:
# § 6 — Controls: status, diagnostics, live mode, kill switch

def status():
    print(f'Running: {BOT_STATE["running"]}  Mode: {CFG["mode"]}  Iter: {BOT_STATE["iter"]}')
    ws_age = ''
    if _WS_STATE.get('last_msg_ts'):
        ws_age = f' (last msg {(datetime.now(timezone.utc) - _WS_STATE["last_msg_ts"]).total_seconds():.0f}s ago)'
    print(f'WS: {"connected" if _WS_STATE["connected"] else "DISCONNECTED"}  '
          f'msgs={_WS_STATE["msg_count"]}  sub={_WS_STATE["subscribed_event"]}{ws_age}')
    with _LOCK:
        spot = SPOT.get('price'); spot_ts = SPOT.get('ts')
        event = TRACKED.get('event'); close = TRACKED.get('close_time')
        refreshed = TRACKED.get('refreshed_at')
        n_books = len(BOOKS)
    if spot:
        age = (datetime.now(timezone.utc) - spot_ts).total_seconds() if spot_ts else None
        if age is not None:
            print(f'Spot: ${spot:,.2f} ({age:.0f}s ago)')
        else:
            print(f'Spot: ${spot:,.2f}')
    else:
        print('Spot: unavailable')
    if event:
        ttl = (close - datetime.now(timezone.utc)).total_seconds() / 60
        print(f'Event: {event}  TTL: {ttl:.1f}min  Books: {n_books}')
    else:
        ref_age = ''
        if refreshed:
            ref_age = f' (tracker last ran {(datetime.now(timezone.utc) - refreshed).total_seconds():.0f}s ago)'
        print(f'Event: NONE TRACKED{ref_age}  — event_tracker has not found an event in the 2min–2hr window')
    sigma = _causal_sigma()
    if sigma:
        print(f'Sigma: {sigma*100:.1f}%')
    else:
        with _LOCK:
            n_hist = len(SPOT.get('history', []))
        print(f'Sigma: insufficient data ({n_hist}/{CFG["sigma_min_points"]} points)')

    sigs = scan_all_signals()
    print(f'\nCurrent signals: T1={len(sigs["tier1"])}  T2={len(sigs["tier2"])}')
    for s in sigs['tier1'][:3]:
        print(f'  T1 {s.mkt_lo} | {s.mkt_hi}  edge={s.net_edge_cents:.1f}c  qty={s.qty}')
    for s in sigs['tier2'][:3]:
        print(f'  T2 {s.ticker} {s.side} @${s.price:.2f}  fair={s.fair_value:.3f}  edge={s.edge_cents:.1f}c')

    pos = _open_positions()
    print(f'\nOpen positions: {len(pos)}')
    for p in pos:
        tag = 'T1' if p['tier'] == 1 else 'T2'
        print(f'  {tag} {p["market_ticker"]} {p["side"]} x{p["contracts"]} @${p["entry_price"]:.2f}')
    print(f'\nToday PnL: ${_today_pnl():+.2f}  Exposure: ${_open_exposure():.2f}')
    print(f'Trades this session: {BOT_STATE["trades_this_session"]}')
    for entry in BOT_STATE['log'][-5:]:
        print(f'  {entry}')


def diagnostics():
    print('=== DIAGNOSTICS ===')
    print(f'Mode: {CFG["mode"]}  Live: {CFG["live_enabled"]}  Iter: {BOT_STATE["iter"]}')
    alive = [t.name for t in BOT_STATE.get('threads', []) if t.is_alive()]
    dead  = [t.name for t in BOT_STATE.get('threads', []) if not t.is_alive()]
    print(f'Threads alive: {alive}')
    if dead: print(f'Threads DEAD: {dead}')
    print(f'\nWS: connected={_WS_STATE["connected"]}  msgs={_WS_STATE["msg_count"]}  '
          f'reconnects={_WS_STATE["reconnect_count"]}')
    ws_last = _WS_STATE.get('last_msg_ts')
    if ws_last:
        age = (datetime.now(timezone.utc) - ws_last).total_seconds()
        print(f'  Last WS msg: {age:.1f}s ago')

    now = datetime.now(timezone.utc)
    with _LOCK:
        spot_age = (now - SPOT['ts']).total_seconds() if SPOT.get('ts') else None
        n_hist = len(SPOT.get('history', []))
        n_books = len(BOOKS)
    spot_age_str = f'{spot_age:.0f}s' if spot_age is not None else '— (no spot yet)'
    print(f'\nSpot age: {spot_age_str}   history: {n_hist} points')
    print(f'Books loaded: {n_books}')

    sigma = _causal_sigma()
    if sigma:
        print(f'Sigma (annual): {sigma*100:.2f}%')
    else:
        print('Sigma: None')

    sigs = scan_all_signals()
    print(f'\nSignals now:  T1={len(sigs["tier1"])}   T2={len(sigs["tier2"])}')
    for s in sigs['tier1'][:5]:
        print(f'  T1 lo={s.mkt_lo} hi={s.mkt_hi}  ask_lo={s.ask_lo:.2f} bid_hi={s.bid_hi:.2f}  '
              f'qty={s.qty} edge={s.net_edge_cents:.1f}c')
    for s in sigs['tier2'][:5]:
        print(f'  T2 {s.ticker} {s.side} @${s.price:.2f}  fair={s.fair_value:.3f}  '
              f'edge={s.edge_cents:.1f}c ttc={s.secs_remaining:.0f}s')

    conn = _trades_conn()
    total = conn.execute('SELECT COUNT(*) FROM trades').fetchone()[0]
    settled = conn.execute('SELECT COUNT(*) FROM trades WHERE settled=1').fetchone()[0]
    pnl = conn.execute('SELECT COALESCE(SUM(pnl),0) FROM trades WHERE settled=1').fetchone()[0]
    wins = conn.execute('SELECT COUNT(*) FROM trades WHERE settled=1 AND pnl>0').fetchone()[0]
    by_tier = conn.execute('SELECT tier, COUNT(*), COALESCE(SUM(pnl),0) FROM trades WHERE settled=1 GROUP BY tier').fetchall()
    conn.close()
    print(f'\nAll-time: total={total} settled={settled}')
    if settled:
        print(f'  PnL=${pnl:+.2f}  Win rate={wins/settled*100:.0f}%')
    for r in by_tier:
        print(f'  Tier {r[0]}: n={r[1]} pnl=${r[2]:+.2f}')

    print(f'\nRecent log:')
    for entry in BOT_STATE['log'][-10:]:
        print(f'  {entry}')


def trade_history(n=20):
    conn = _trades_conn()
    rows = conn.execute('SELECT * FROM trades ORDER BY id DESC LIMIT ?', (n,)).fetchall()
    conn.close()
    if not rows:
        print('No trades yet.'); return
    df = pd.DataFrame([dict(r) for r in rows])
    cols = ['id', 'ts', 'tier', 'pair_id', 'market_ticker', 'side', 'contracts',
            'entry_price', 'edge_cents', 'mode', 'settled', 'settlement_result', 'pnl']
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))


def enable_live():
    if kalshi_live is None:
        print('No prod credentials — cannot go live.'); return
    try:
        bal = kalshi_live.get_balance()
        balance = float(bal.get('balance', 0)) / 100.0
    except Exception as e:
        print(f'Balance check failed: {e}'); return
    if balance <= 0:
        print('Balance $0 — refusing.'); return
    CFG['mode'] = 'live'; CFG['live_enabled'] = True
    print(f'LIVE TRADING ENABLED.  Balance ${balance:.2f}')
    print(f'  Max exposure: ${CFG["max_total_exposure"]}  Daily loss cap: ${CFG["daily_loss_limit"]}')


def disable_live():
    CFG['mode'] = 'paper'; CFG['live_enabled'] = False
    print('Switched to paper mode.')


def kill_switch():
    print('KILL SWITCH activated')
    CFG['mode'] = 'paper'; CFG['live_enabled'] = False
    stop_bot()
    if kalshi_live:
        try:
            orders = kalshi_live._get('/portfolio/orders', {'status': 'resting'}).get('orders', []) or []
            for o in orders:
                oid = o.get('order_id')
                if oid:
                    kalshi_live.cancel_order(oid)
            print(f'  Cancelled {len(orders)} resting orders')
        except Exception as e:
            print(f'  cancel err: {e}')


print('Controls ready.')
print('  status()         — quick state')
print('  diagnostics()    — detailed')
print('  trade_history()  — recent trades')
print('  tick_stats()     — data collection stats')
print('  enable_live()    — switch paper -> live')
print('  disable_live()   — back to paper')
print('  kill_switch()    — halt + cancel all')


Controls ready.
  status()         — quick state
  diagnostics()    — detailed
  trade_history()  — recent trades
  tick_stats()     — data collection stats
  enable_live()    — switch paper -> live
  disable_live()   — back to paper
  kill_switch()    — halt + cancel all


In [15]:
# § 6b — think() — verbose signal explainer
#
# Walks every market in BOOKS and prints WHY each is or isn't a signal.
# Used to debug "the bot isn't trading" — shows you exactly what it sees.


def think(max_rows=15):
    """Detailed dump of what the strategy is seeing right now."""
    with _LOCK:
        spot = SPOT.get('price')
        spot_ts = SPOT.get('ts')
        event = TRACKED.get('event')
        close_time = TRACKED.get('close_time')
        snap = {k: dict(v) for k, v in BOOKS.items()}

    print('═══════════════════════════════════════════════════════════════════════')
    print(' BOT REASONING — what the strategy sees right now')
    print('═══════════════════════════════════════════════════════════════════════')

    # ── Market state ─────────────────────────────────────────────
    print(f'\n[ MARKET STATE ]')
    print(f'  Spot: ${spot:,.2f}' if spot else '  Spot: ─ (no data)')
    if spot_ts:
        print(f'  Spot age: {(datetime.now(timezone.utc) - spot_ts).total_seconds():.1f}s')
    print(f'  Event:    {event or "─ (none tracked)"}')
    if close_time:
        ttc = (close_time - datetime.now(timezone.utc)).total_seconds()
        print(f'  Close in: {ttc:.0f}s ({ttc/60:.1f}min)')
    print(f'  Books loaded: {len(snap)}')
    sigma = _causal_sigma()
    if sigma:
        sig_s = sigma / math.sqrt(365.25 * 24 * 3600)
        sig_rem = sig_s * spot * math.sqrt(max(1, (close_time - datetime.now(timezone.utc)).total_seconds())) if (spot and close_time) else None
        print(f'  Sigma (annual): {sigma*100:.2f}%   1-σ move to expiry: ${sig_rem:.0f}' if sig_rem else f'  Sigma (annual): {sigma*100:.2f}%')
    else:
        with _LOCK:
            n_hist = len(SPOT.get('history', []))
        print(f'  Sigma: insufficient ({n_hist}/{CFG["sigma_min_points"]} spot points)')

    # ── Universe: active markets sorted by strike ────────────────
    # Apply the SAME filter the scanner uses, so what you see matches reality
    active = []
    skipped_low_bid = 0
    for tk, b in snap.items():
        if b.get('status') != 'active' or b.get('floor') is None: continue
        ya = b.get('yes_ask'); yb = b.get('yes_bid')
        if ya is None or yb is None: continue
        if yb <= 0.01 or ya >= 0.99:
            skipped_low_bid += 1
            continue
        active.append({'tk': tk, 'K': float(b['floor']),
                       'ya': ya, 'yb': yb,
                       'ya_qty': b.get('yes_ask_qty') or 0,
                       'yb_qty': b.get('yes_bid_qty') or 0})
    active.sort(key=lambda r: r['K'])

    # Contract-type sanity check: threshold contracts have monotone yes_ask in strike
    # (decreasing as K increases). Bell-curve = range/bucket → strategy doesn't apply.
    contract_type = 'threshold'
    if event and not event.startswith('KXBTCD'):
        contract_type = 'NOT-KXBTCD'
    elif len(active) >= 10:
        atm_K = spot if spot else (active[len(active)//2]['K'])
        below = [r for r in active if r['K'] < atm_K]
        above = [r for r in active if r['K'] >= atm_K]
        if below and above:
            avg_below_ask = sum(r['ya'] for r in below) / len(below)
            avg_above_ask = sum(r['ya'] for r in above) / len(above)
            if avg_below_ask < avg_above_ask:
                contract_type = 'BELL (range/bucket)'

    print(f'\n[ UNIVERSE ]  {len(active)} active markets with real bids '
          f'(skipped {skipped_low_bid} with yes_bid≤0.01 or yes_ask≥0.99)')
    if contract_type != 'threshold':
        print(f'  ⚠️  CONTRACT TYPE: {contract_type}')
        print(f'  ⚠️  Strategy assumes THRESHOLD ("BTC ≥ $K"). This event prices like a')
        print(f'  ⚠️  range/bucket contract — Tier 1 and Tier 2 should NOT trade here.')
        print(f'  ⚠️  Make sure event_series = ("KXBTCD",) — current = {CFG["event_series"]}')
    if active and spot:
        atm_idx = min(range(len(active)), key=lambda i: abs(active[i]['K'] - spot))
        lo = max(0, atm_idx - 5); hi = min(len(active), atm_idx + 6)
        print(f'  Strikes near spot (${spot:,.0f}):')
        print(f'    {"strike":>10s} {"yes_bid":>10s} {"yes_ask":>10s} {"spread":>8s} {"bid_qty":>8s} {"ask_qty":>8s}')
        for i in range(lo, hi):
            r = active[i]
            arrow = ' ←ATM' if i == atm_idx else ''
            spr = (r['ya'] - r['yb']) * 100
            print(f'    {r["K"]:>10,.0f} {r["yb"]:>10.2f} {r["ya"]:>10.2f} {spr:>7.1f}¢ '
                  f'{int(r["ya_qty"]):>8} {int(r["yb_qty"]):>8}{arrow}')

    # ── Tier 1: monotonicity — show ALL pairs where bid_hi > ask_lo ──
    print(f'\n[ TIER 1 — Monotonicity Arb ]')
    print(f'  Rule: yes_bid(K_hi) > yes_ask(K_lo) + fees → risk-free pair trade')
    print(f'  Threshold: min_net_edge ≥ {CFG["t1_min_net_edge_cents"]}¢')

    t1_all = []  # all pairs with gross > 0
    for i, lo in enumerate(active):
        for hi in active[i+1:]:
            if hi['yb'] > lo['ya']:
                gross = hi['yb'] - lo['ya']
                fees = kalshi_fee(lo['ya']) + kalshi_fee(1.0 - hi['yb'])
                net = gross - fees
                t1_all.append({
                    'lo': lo['tk'], 'hi': hi['tk'],
                    'K_lo': lo['K'], 'K_hi': hi['K'],
                    'ask_lo': lo['ya'], 'bid_hi': hi['yb'],
                    'gross_c': gross * 100, 'fees_c': fees * 100, 'net_c': net * 100,
                    'qty': min(lo['ya_qty'] or 1, hi['yb_qty'] or 1, CFG['t1_max_qty_per_leg']),
                    'passes': net >= CFG['t1_min_net_edge_cents'] / 100,
                })

    if not t1_all:
        print('  → No monotonicity violations in the book (this is normal — markets are usually consistent)')
    else:
        t1_all.sort(key=lambda r: r['net_c'], reverse=True)
        passing = [r for r in t1_all if r['passes']]
        print(f'  Pairs with bid_hi > ask_lo: {len(t1_all)}  |  passing fee threshold: {len(passing)}')
        print(f'  Top {min(max_rows, len(t1_all))}:')
        print(f'    {"K_lo":>8s} {"K_hi":>8s} {"ask_lo":>7s} {"bid_hi":>7s} {"gross":>6s} {"fees":>5s} {"net":>6s} {"qty":>4s}  status')
        for r in t1_all[:max_rows]:
            tag = '✓ TRADE' if r['passes'] else f'✗ net {r["net_c"]:+.1f}¢ < {CFG["t1_min_net_edge_cents"]}¢'
            print(f'    {r["K_lo"]:>8,.0f} {r["K_hi"]:>8,.0f} {r["ask_lo"]:>7.2f} {r["bid_hi"]:>7.2f} '
                  f'{r["gross_c"]:>5.1f}¢ {r["fees_c"]:>4.1f}¢ {r["net_c"]:>+5.1f}¢ {r["qty"]:>4}  {tag}')

    # ── Tier 2: deep-ITM — show fair value vs market price per strike ──
    print(f'\n[ TIER 2 — Deep-ITM Convergence ]')
    print(f'  Rule: price ≥ {CFG["t2_min_price"]}, fair ≥ {CFG["t2_min_fair"]}, edge ≥ {CFG["t2_min_edge_cents"]}¢, '
          f'TTC ∈ [{CFG["t2_min_secs_to_close"]}, {CFG["t2_max_secs_to_close"]}]s')

    if not spot or not close_time:
        print('  → No spot or close_time — skipping')
    elif not sigma:
        print('  → No sigma yet — skipping (need ≥15 spot points)')
    else:
        secs = (close_time - datetime.now(timezone.utc)).total_seconds()
        if not (CFG['t2_min_secs_to_close'] <= secs <= CFG['t2_max_secs_to_close']):
            print(f'  → TTC {secs:.0f}s is outside [{CFG["t2_min_secs_to_close"]}, {CFG["t2_max_secs_to_close"]}]s — Tier 2 idle')
        else:
            sigma_f = max(sigma, CFG['t2_sigma_floor_annual'])
            sigma_c = sigma_f * (1.0 + CFG['sigma_uncertainty_discount'])
            sig_s = sigma_c / math.sqrt(365.25 * 24 * 3600)
            sig_rem = sig_s * spot * math.sqrt(secs)
            min_dist = max(CFG['t2_min_dollar_distance_from_strike'],
                           spot * CFG['t2_min_pct_distance_from_strike'])
            print(f'  σ_rem (1-σ move to expiry): ${sig_rem:.0f}   (floored σ {sigma_f*100:.1f}%, +disc {CFG["sigma_uncertainty_discount"]*100:.0f}%)')
            print(f'  Min |spot − K|: ${min_dist:.0f} (max ${CFG["t2_min_dollar_distance_from_strike"]:.0f} or {CFG["t2_min_pct_distance_from_strike"]*100:.1f}%)')
            print(f'  Per-strike fair vs market (sorted by |edge|):')
            rows = []
            for r in active:
                K = r['K']
                dist = abs(spot - K)
                d = dist / sig_rem
                fair_yes = _norm_cdf(d) if spot > K else 1.0 - _norm_cdf(d)
                fair_no = 1.0 - fair_yes
                yes_edge = fair_yes - r['ya'] - kalshi_fee(r['ya'])
                no_price = 1.0 - r['yb']
                no_edge = fair_no - no_price - kalshi_fee(no_price)
                for side, price, fair, edge in [('yes', r['ya'], fair_yes, yes_edge),
                                                 ('no',  no_price, fair_no, no_edge)]:
                    rows.append({
                        'K': K, 'side': side, 'price': price, 'fair': fair,
                        'edge_c': edge * 100, 'dist': dist,
                        'passes': (price >= CFG['t2_min_price'] and price <= 0.97
                                   and fair >= CFG['t2_min_fair']
                                   and edge >= CFG['t2_min_edge_cents'] / 100
                                   and dist >= min_dist),
                    })
            rows.sort(key=lambda x: x['edge_c'], reverse=True)
            passing = [r for r in rows if r['passes']]
            print(f'  Total candidates: {len(rows)}  |  passing all filters: {len(passing)}')
            print(f'    {"strike":>8s} {"side":>4s} {"price":>6s} {"fair":>6s} {"edge":>7s} {"dist":>6s}  status')
            for r in rows[:max_rows]:
                reasons = []
                if r['price'] < CFG['t2_min_price']: reasons.append(f'price<{CFG["t2_min_price"]}')
                if r['price'] > 0.97: reasons.append('price>0.97')
                if r['fair'] < CFG['t2_min_fair']: reasons.append(f'fair<{CFG["t2_min_fair"]}')
                if r['edge_c'] < CFG['t2_min_edge_cents']: reasons.append(f'edge<{CFG["t2_min_edge_cents"]}¢')
                if r['dist'] < min_dist: reasons.append(f'dist<${min_dist:.0f}')
                tag = '✓ TRADE' if r['passes'] else ('✗ ' + ', '.join(reasons) if reasons else '✗')
                print(f'    {r["K"]:>8,.0f} {r["side"]:>4s} {r["price"]:>6.2f} {r["fair"]:>6.3f} '
                      f'{r["edge_c"]:>+6.1f}¢ ${r["dist"]:>5.0f}  {tag}')

    # ── Risk gate state ──────────────────────────────────────────
    print(f'\n[ RISK GATES ]')
    open_pos = _open_positions()
    exp = _open_exposure()
    pnl = _today_pnl()
    print(f'  Open positions: {len(open_pos)} / {CFG["max_concurrent_positions"]}')
    print(f'  Exposure:       ${exp:.2f} / ${CFG["max_total_exposure"]:.0f}')
    print(f'  Today PnL:      ${pnl:+.2f}  (daily loss limit ${CFG["daily_loss_limit"]:.0f})')
    for p in open_pos:
        tag = 'T1' if p['tier'] == 1 else 'T2'
        print(f'    {tag} {p["market_ticker"]} {p["side"]} x{p["contracts"]} @${p["entry_price"]:.2f}')

    # ── What the scanners ACTUALLY return (after filters) ────────
    sigs = scan_all_signals()
    print(f'\n[ EXECUTIONABLE SIGNALS NOW ]  T1={len(sigs["tier1"])}  T2={len(sigs["tier2"])}')
    for s in sigs['tier1'][:5]:
        print(f'  T1 BUY {s.mkt_lo} yes @{s.ask_lo:.2f} + BUY {s.mkt_hi} no @{1-s.bid_hi:.2f}  '
              f'edge={s.net_edge_cents:.1f}¢ qty={s.qty}')
    for s in sigs['tier2'][:5]:
        print(f'  T2 BUY {s.ticker} {s.side} x{s.qty} @${s.price:.2f}  '
              f'fair={s.fair_value:.3f} edge={s.edge_cents:.1f}¢')

    print('═══════════════════════════════════════════════════════════════════════')


print('think() ready — call to see exactly what each tier is evaluating.')


think() ready — call to see exactly what each tier is evaluating.


In [16]:
# § 6c — Live-updating portfolio dashboard
#
# Mark-to-market every open position using current order book, show realized
# + unrealized PnL, refresh every N seconds without re-running the cell.
# Stop with Jupyter's ■ (interrupt kernel).


def _market_to_market(pos):
    """Return (current_mark, unrealized_pnl_per_contract, source) for one position.

    Conservative: uses the price we could exit at right now (bid for our side).
    For a YES long, exit = yes_bid. For a NO long, exit = no_bid = 1 - yes_ask.
    Returns (None, None, 'stale') if no quote available."""
    with _LOCK:
        b = dict(BOOKS.get(pos['market_ticker'], {}))
    if not b:
        return None, None, 'no book'
    side = pos['side']
    entry = pos['entry_price']
    if side == 'yes':
        exit_p = b.get('yes_bid')
    else:
        ya = b.get('yes_ask')
        exit_p = (1.0 - ya) if ya is not None else None
    if exit_p is None:
        return None, None, 'no quote'
    # PnL if we closed now: payout (= exit_p) - cost (= entry + fee)
    # No exit fee assumed for now (Kalshi sometimes waives close-side fee for makers)
    unrealized = exit_p - entry - kalshi_fee(entry)
    return exit_p, unrealized, 'live'


def _pair_unrealized(pos_a, pos_b):
    """For a Tier 1 pair, compute combined unrealized PnL."""
    m_a, u_a, src_a = _market_to_market(pos_a)
    m_b, u_b, src_b = _market_to_market(pos_b)
    if u_a is None or u_b is None:
        return None, src_a if u_a is None else src_b
    total = (u_a + u_b) * pos_a['contracts']  # both legs have same qty
    return total, 'live'


def _render_dashboard():
    out = []
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    mode_str = 'LIVE' if CFG['mode'] == 'live' else 'PAPER'
    out.append(f'═════════ KALSHI ARB BOT ═════════ {now} ═════════ Mode: {mode_str} ═════════')

    # ── Connection / data freshness ─────────────────────────────
    ws_age = ''
    if _WS_STATE.get('last_msg_ts'):
        ws_age = f' (last {(datetime.now(timezone.utc) - _WS_STATE["last_msg_ts"]).total_seconds():.0f}s ago)'
    ws_ok = '✓' if _WS_STATE.get('connected') else '✗'
    out.append(f'WS {ws_ok} msgs={_WS_STATE["msg_count"]}  sub={_WS_STATE.get("subscribed_event") or "—"}{ws_age}')

    with _LOCK:
        spot = SPOT.get('price'); spot_ts = SPOT.get('ts')
        event = TRACKED.get('event'); close = TRACKED.get('close_time')
        n_books = len(BOOKS)
    spot_line = 'Spot: ─'
    if spot is not None:
        age = (datetime.now(timezone.utc) - spot_ts).total_seconds() if spot_ts else None
        spot_line = f'Spot: ${spot:,.2f}' + (f' ({age:.0f}s ago)' if age is not None else '')
    out.append(spot_line)
    if event:
        ttc = (close - datetime.now(timezone.utc)).total_seconds() / 60 if close else None
        ttc_str = f'  TTL: {ttc:.1f}min' if ttc is not None else ''
        out.append(f'Event: {event}{ttc_str}  Books: {n_books}')
    else:
        out.append(f'Event: — (no active event in window)')
    sigma = _causal_sigma()
    if sigma:
        out.append(f'Sigma: {sigma*100:.1f}% annual')
    else:
        with _LOCK:
            n_hist = len(SPOT.get('history', []))
        out.append(f'Sigma: warming up ({n_hist}/{CFG["sigma_min_points"]} pts)')

    # ── Portfolio ───────────────────────────────────────────────
    open_pos = _open_positions()
    realized_today = _today_pnl()
    realized_all_time = 0.0
    conn = _trades_conn()
    r = conn.execute('SELECT COALESCE(SUM(pnl),0), COUNT(*) FILTER (WHERE settled=1) FROM trades').fetchone()
    realized_all_time = float(r[0])
    n_settled = int(r[1])
    conn.close()

    out.append('')
    out.append(f'┌─ PORTFOLIO ─────────────────────────────────────────────────────────────────')

    # Group T1 positions by pair_id; T2/T3 are single-leg
    t1_pairs = {}
    single_positions = []  # T2 + T3
    for p in open_pos:
        if p['tier'] == 1 and p.get('pair_id'):
            t1_pairs.setdefault(p['pair_id'], []).append(p)
        else:
            single_positions.append(p)

    n_t2 = sum(1 for p in single_positions if p['tier'] == 2)
    n_t3 = sum(1 for p in single_positions if p['tier'] == 3)
    unrealized_total = 0.0
    out.append(f'│ Open positions: {len(open_pos)} / {CFG["max_concurrent_positions"]}'
               f'  (T1 pairs: {len(t1_pairs)}, T2: {n_t2}, T3: {n_t3})')

    # Tier 1 pairs
    if t1_pairs:
        out.append(f'│')
        out.append(f'│ Tier 1 pairs (risk-free arbitrage):')
        for pid, legs in t1_pairs.items():
            if len(legs) != 2:
                out.append(f'│   ⚠ orphan pair {pid[-6:]}  ({len(legs)} legs)')
                continue
            u, src = _pair_unrealized(legs[0], legs[1])
            if u is not None:
                unrealized_total += u
            u_str = f'${u:+.2f}' if u is not None else '—'
            out.append(f'│   {pid[-8:]}  {legs[0]["market_ticker"][-18:]} {legs[0]["side"]:3s}'
                       f' / {legs[1]["market_ticker"][-18:]} {legs[1]["side"]:3s}'
                       f'  x{legs[0]["contracts"]}  unreal={u_str}')

    # T2/T3 single legs
    if single_positions:
        out.append(f'│')
        out.append(f'│ Single-leg positions (mark-to-market):')
        out.append(f'│   {"tier":>4s} {"market_ticker":<26s} {"side":>4s} {"qty":>3s} {"entry":>6s} '
                   f'{"mark":>6s} {"unreal":>8s}')
        for p in single_positions:
            mark, u_per_c, src = _market_to_market(p)
            u = (u_per_c * p['contracts']) if u_per_c is not None else None
            if u is not None:
                unrealized_total += u
            mark_str = f'${mark:.2f}' if mark is not None else '—'
            u_str = f'${u:+.2f}' if u is not None else '—'
            tk_short = p['market_ticker'][-26:]
            tier_str = f'T{p["tier"]}'
            out.append(f'│   {tier_str:>4s} {tk_short:<26s} {p["side"]:>4s} {p["contracts"]:>3} '
                       f'${p["entry_price"]:>5.2f} {mark_str:>6s} {u_str:>8s}')

    if not open_pos:
        out.append(f'│ (no open positions)')

    # Paper account snapshot (paper mode only)
    try:
        if CFG['mode'] == 'paper':
            pa = PAPER_ACCOUNT
            mtm = paper_mark_to_market()
            ret = (mtm - pa['starting_cash']) / pa['starting_cash'] * 100
            out.append(f'│')
            out.append(f'│ ── PAPER ACCOUNT (${pa["starting_cash"]:.2f} start) ──')
            out.append(f'│   Cash: ${pa["cash"]:.2f}  Locked: ${pa["locked"]:.2f}  '
                       f'MtM: ${mtm:.2f}  Return: {ret:+.2f}%')
            wpct = (pa["wins"] / pa["trades_settled"] * 100) if pa["trades_settled"] else 0
            out.append(f'│   Settled: {pa["trades_settled"]} (W{pa["wins"]}/L{pa["losses"]} = {wpct:.0f}%)  '
                       f'MaxDD: {pa["max_drawdown"]*100:.1f}%')
    except NameError:
        pass

    # PnL summary
    out.append(f'│')
    net_today = realized_today + unrealized_total
    pnl_color_today = '+' if net_today >= 0 else ''
    out.append(f'│ Realized today: ${realized_today:+8.2f}    '
               f'Unrealized: ${unrealized_total:+8.2f}    '
               f'Net today: ${pnl_color_today}{net_today:+8.2f}')
    out.append(f'│ All-time realized: ${realized_all_time:+8.2f} across {n_settled} settled trades')
    out.append(f'│ Exposure: ${_open_exposure():.2f} / ${CFG["max_total_exposure"]:.0f}')
    out.append(f'└─────────────────────────────────────────────────────────────────────────────')

    # ── Active signals (right now) ──────────────────────────────
    sigs = scan_all_signals()
    out.append('')
    out.append(f'┌─ LIVE SIGNALS (this scan) ──────────────────────────────────────────────────')
    out.append(f'│ T1={len(sigs["tier1"])} T2={len(sigs["tier2"])} T3={len(sigs["tier3"])}  '
               f'(actionable right now, after pair-dedup)')
    for s in sigs['tier1'][:2]:
        out.append(f'│   T1 {s.mkt_lo[-18:]:18s} yes@{s.ask_lo:.2f} + '
                   f'{s.mkt_hi[-18:]:18s} no@{1-s.bid_hi:.2f}  '
                   f'edge={s.net_edge_cents:.1f}¢ qty={s.qty}')
    for s in sigs['tier2'][:2]:
        out.append(f'│   T2 {s.ticker[-26:]:26s} {s.side:3s} x{s.qty} @${s.price:.2f}  '
                   f'fair={s.fair_value:.3f} edge={s.edge_cents:.1f}¢')
    for s in sigs['tier3'][:2]:
        out.append(f'│   T3 {s.ticker[-26:]:26s} no  x{s.qty} @${s.no_price:.2f}  '
                   f'dist=${s.spot_dist:.0f} persist={s.persistence_min:.0f}min')
    if not (sigs['tier1'] or sigs['tier2'] or sigs['tier3']):
        out.append('│   (no actionable signals right now)')
    out.append(f'└──────────────────────────────────────────────────────────────────────────────')

    # ── Signal flow telemetry (session totals + most-recent activity) ──────
    lc = SIGNAL_FLOW['last_cycle']
    tot = SIGNAL_FLOW['totals']
    cyc = SIGNAL_FLOW['cycles']
    out.append('')
    out.append(f'┌─ SIGNAL FLOW ({cyc} scan cycles this session) ───────────────────────────────')
    out.append(f'│ Last cycle:  T1 seen={lc["t1_seen"]} exec={lc["t1_exec"]}  '
               f'T2 seen={lc["t2_seen"]} exec={lc["t2_exec"]}  '
               f'T3 seen={lc["t3_seen"]} exec={lc["t3_exec"]}')
    out.append(f'│ Session sum: T1 seen={tot["t1_seen"]} exec={tot["t1_exec"]}  '
               f'T2 seen={tot["t2_seen"]} exec={tot["t2_exec"]}  '
               f'T3 seen={tot["t3_seen"]} exec={tot["t3_exec"]}')
    if SIGNAL_FLOW['skip_reasons']:
        top_reasons = sorted(SIGNAL_FLOW['skip_reasons'].items(),
                              key=lambda x: -x[1])[:5]
        out.append(f'│ Top skip reasons: ' + ', '.join(f'{r}={n}' for r, n in top_reasons))
    out.append(f'│')
    out.append(f'│ Recent signal decisions (latest 8):')
    if SIGNAL_FLOW['recent']:
        for entry in list(SIGNAL_FLOW['recent'])[-8:]:
            ts, tier, ticker, status, detail = entry
            mark = '✓' if status == 'EXEC' else '✗'
            out.append(f'│   [{ts}] {mark} T{tier} {ticker:<28s} {status:<25s} {detail}')
    else:
        out.append(f'│   (no signal activity yet)')
    out.append(f'└──────────────────────────────────────────────────────────────────────────────')

    # ── SPRT (auto-disable monitor) ────────────────────────────
    try:
        out.append('')
        out.append(f'┌─ SPRT MONITOR — auto-disable when win rate drops below target ──────────────')
        for tier in (1, 2, 3):
            st = sprt_state(tier)
            if not st:
                continue
            status = 'DISABLED' if st['disabled'] else 'active'
            verdict = ''
            if st['llr'] <= SPRT_LOWER:
                verdict = '← DISABLE'
            elif st['llr'] >= SPRT_UPPER:
                verdict = '← CONFIRMED'
            out.append(f'│   T{tier}: {status:>8s}  n={st["n"]:>3}  '
                       f'LLR={st["llr"]:+6.2f}  '
                       f'(target {st["p_target"]*100:.0f}% / breakeven {st["p_be"]*100:.0f}%) {verdict}')
        out.append(f'└──────────────────────────────────────────────────────────────────────────────')
    except NameError:
        pass

    # ── Recent activity ─────────────────────────────────────────
    out.append('')
    out.append('Recent activity:')
    for entry in BOT_STATE['log'][-5:]:
        out.append(f'  {entry}')

    return '\n'.join(out)


def live_status(refresh_sec=2):
    """Auto-refreshing dashboard. Press ■ (interrupt kernel) to stop."""
    try:
        from IPython.display import clear_output
        in_jupyter = True
    except ImportError:
        in_jupyter = False
        def clear_output(wait=True):
            os.system('clear' if os.name == 'posix' else 'cls')

    try:
        while True:
            clear_output(wait=True)
            print(_render_dashboard())
            print(f'\n[refreshing every {refresh_sec}s — press ■ stop to halt]')
            time.sleep(refresh_sec)
    except KeyboardInterrupt:
        clear_output(wait=True)
        print(_render_dashboard())
        print('\n[dashboard stopped]')


print('Live dashboard ready.')
print('  live_status()        — refreshes every 2s with mark-to-market PnL')
print('  live_status(5)       — refresh every 5s')


Live dashboard ready.
  live_status()        — refreshes every 2s with mark-to-market PnL
  live_status(5)       — refresh every 5s


## Run Book

### Paper trading (default)
1. Run every code cell above (§1 → §6)
2. `start_bot()` — fires up websocket, polls spot, scans signals, places paper trades
3. `status()` — quick snapshot (signals, positions, PnL)
4. `diagnostics()` — full detail (threads, sigma, recent log)
5. `stop_bot()` — when done

### Live trading
1. Verify with paper for ≥1 hour first
2. `enable_live()` — flips mode + sanity-checks balance
3. `status()` — monitor
4. `kill_switch()` — instant stop (cancels all resting orders)

### Data collection
The bot logs every websocket tick to `output/arb_strategy_ticks.db`. After collecting a
few weeks of data, recalibrate Tier 2 thresholds against fresh ticks.

### How the strategy works

**Tier 1** is mathematically risk-free. When the books show `yes_bid(K_hi) > yes_ask(K_lo)`,
buy YES at the low strike and (equivalently) sell YES at the high = buy NO at the high.
For ANY settlement outcome the net is at least `bid_hi - ask_lo - 2×fees`, which is positive
by construction.

**Tier 2** is statistical. Buy deep-ITM contracts where Black-Scholes binary fair value
exceeds the market price by enough to overcome Kalshi's 7% fee. Strict filters
(`price ≥ 0.88, fair ≥ 0.94, TTC ∈ [15min, 60min]`) ensure positive expected value.


In [17]:
# Start in paper mode (default)
start_bot()


Bootstrapping: looking for active event...
  → tracking KXBTCD-26MAY1717 closes 2026-05-17 21:00:00+00:00 (from 3 candidates)
Bot started.  Mode=paper  T1=True  T2=True
  Tier1: min_edge=1.5c max_qty=20
  Tier2: price>=0.88 fair>=0.95 ttc=[1800,3600]s
  Risk: max_pos=4  exp_cap=$200.0  daily_loss=$-30.0
  status()  diagnostics()  tick_stats()  stop_bot()


In [18]:
# Live-updating portfolio dashboard with mark-to-market PnL.
# Refreshes every 2s. Press ■ (stop) in Jupyter to halt.
live_status()


═════════ KALSHI ARB BOT ═════════ 20:18:55 UTC ═════════ Mode: PAPER ═════════
WS ✓ msgs=2643  sub=KXBTCD-26MAY1717 (last 1s ago)
Spot: $78,291.04 (1s ago)
Event: KXBTCD-26MAY1717  TTL: 41.1min  Books: 80
Sigma: 3.7% annual

┌─ PORTFOLIO ─────────────────────────────────────────────────────────────────
│ Open positions: 0 / 4  (T1 pairs: 0, T2: 0, T3: 0)
│ (no open positions)
│
│ ── PAPER ACCOUNT ($100.00 start) ──
│   Cash: $101.20  Locked: $0.00  MtM: $101.20  Return: +1.20%
│   Settled: 4.0 (W3.0/L1.0 = 75%)  MaxDD: 0.9%
│
│ Realized today: $   +0.05    Unrealized: $   +0.00    Net today: $+   +0.05
│ All-time realized: $   +1.20 across 4 settled trades
│ Exposure: $0.00 / $200
└─────────────────────────────────────────────────────────────────────────────

┌─ LIVE SIGNALS (this scan) ──────────────────────────────────────────────────
│ T1=0 T2=0 T3=0  (actionable right now, after pair-dedup)
│   (no actionable signals right now)
└────────────────────────────────────────────────────

In [ ]:
# One-shot snapshot (non-blocking)
status()


In [ ]:
# Verbose explainer — every candidate the strategy sees + why
think()


In [ ]:
diagnostics()


In [ ]:
tick_stats()


In [ ]:
# After an overnight run: lock in a backtestable snapshot of the ticks DB.
# Then in shell:  python backtest_outputs/merge_overnight_capture.py <path>
# snapshot_ticks_db("night1")


In [ ]:
# Paper account snapshot (cash, locked, realized PnL, win rate, drawdown)
paper_stats()


In [ ]:
# To wipe paper account and restart with $100:
# paper_reset()


In [ ]:
trade_history()


In [ ]:
stop_bot()


In [ ]:
# Uncomment to go live (after verifying paper mode):
# enable_live()
# start_bot()


In [ ]:
# Emergency stop:
# kill_switch()
